# Self-Employed Business Density

**Project:** Suburban Futures (MDAP, University of Melbourne)  
**Purpose:** Build a Greater Melbourne SA2 map of non-employing businesses per 1,000 working-age residents, with industry filters, a 2019-2024 time slider, Story Mode walkthrough, and a 2024 static preview.

Workflow:

1. Inspect the ABS source workbooks.
2. Build the tidy self-employed business density analysis CSV from CABEE and ERP inputs.
3. Build the interactive Plotly HTML map, industry-filter CSV, Story Mode walkthrough, and static 2024 PNG preview.

Local data and generated outputs live outside Git, usually in the project Mediaflux collection.

## Inspect Source Files — Setup

This setup cell locates the downloaded ABS workbooks and defines helper functions for inspecting raw sheet structure. This inspection section is intentionally read-only: it prints workbook metadata, likely header rows, SA2 naming examples, and methodology clues without writing any processed data.

In [ ]:

from pathlib import Path
import re
import traceback

import pandas as pd
from openpyxl import load_workbook

CWD = Path.cwd().resolve()
if (CWD / "data").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data").exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError("Could not find the repo root containing the data/ directory.")

FILES = {
    'CABEE 2021-2025': REPO_ROOT / 'data' / 'raw' / 'abs' / 'cabee' / 'cabee_sa2_industry_employment_size_2021_2025.xlsx',
    'CABEE 2018-2022': REPO_ROOT / 'data' / 'raw' / 'abs' / 'cabee' / 'cabee_sa2_industry_employment_size_2018_2022.xlsx',
    'ERP by SA2 age/sex 2001-2024': REPO_ROOT / 'data' / 'raw' / 'abs' / 'population_age_sex' / 'erp_sa2_age_sex_2001_2024.xlsx',
}
SEARCH_TERMS = [
    'Tarneit', 'Cranbourne', 'Wyndham', 'Melton', 'Mickleham',
    'Melbourne CBD', 'Southbank', 'Docklands'
]

def size_mb(path):
    return path.stat().st_size / (1024 * 1024)

def workbook_sheet_dimensions(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    dims = []
    for ws in wb.worksheets:
        dims.append((ws.title, ws.max_row, ws.max_column))
    wb.close()
    return dims

def print_file_overview(path):
    print(f'Full path: {path}')
    print(f'File size: {size_mb(path):.2f} MB')
    print('Sheet names and dimensions:')
    for sheet, rows, cols in workbook_sheet_dimensions(path):
        print(f'  - {sheet}: {rows:,} rows x {cols:,} columns')

def print_raw_rows(path, sheet_name, n=15):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=n, dtype=object, engine='openpyxl')
    print(f'First {n} raw rows from sheet {sheet_name!r}:')
    print(raw.to_string(index=True, header=False))

def flatten_columns(columns):
    flattened = []
    for col in columns:
        if isinstance(col, tuple):
            parts = [str(x).strip() for x in col if pd.notna(x) and not str(x).startswith('Unnamed')]
            if len(parts) >= 2 and parts[-1].lower() in {'no.', 'no', 'number'}:
                flattened.append(parts[0])
            elif parts:
                flattened.append(' '.join(parts))
            else:
                flattened.append('')
        else:
            flattened.append(str(col).strip())
    return flattened

def load_cabee_sheet(path, sheet_name):
    df = pd.read_excel(path, sheet_name=sheet_name, header=[4, 5], dtype=object, engine='openpyxl')
    df.columns = flatten_columns(df.columns)
    return df

def load_erp_sheet(path, sheet_name):
    df = pd.read_excel(path, sheet_name=sheet_name, header=6, dtype=object, engine='openpyxl')
    df.columns = [str(c).strip() for c in df.columns]
    return df

def clean_cabee_data(df):
    out = df.copy()
    out = out[pd.to_numeric(out['SA2 Code'], errors='coerce').notna()].copy()
    return out

def clean_erp_data(df):
    out = df.copy()
    out = out[pd.to_numeric(out['SA2 code'], errors='coerce').notna()].copy()
    return out

def print_loaded_df_details(df, label):
    print(f'{label} loaded dataframe column names:')
    print(list(df.columns))
    print('\nDtypes:')
    print(df.dtypes.to_string())
    print(f'\nLoaded dataframe row count: {len(df):,}')
    print('\nhead(5):')
    print(df.head(5).to_string(index=False))

def print_sa2_samples(df, sa2_col):
    mask = pd.Series(False, index=df.index)
    for term in SEARCH_TERMS:
        mask = mask | df[sa2_col].astype(str).str.contains(term, case=False, na=False, regex=False)
    samples = sorted(df.loc[mask, sa2_col].dropna().astype(str).unique())[:10]
    print(f'SA2 sample names containing search terms ({sa2_col}):')
    if samples:
        for name in samples:
            print(f'  - {name}')
    else:
        print('  No matches found.')

def cabee_years_from_sheets(path):
    years = []
    wb = load_workbook(path, read_only=True, data_only=True)
    for ws in wb.worksheets:
        # CABEE data sheets have the data year in row 4 table titles.
        # This avoids picking up release-range years from the workbook banner.
        title_parts = []
        for row in ws.iter_rows(min_row=4, max_row=4, values_only=True):
            title_parts.extend(str(v) for v in row if v is not None)
        title = ' '.join(title_parts)
        if 'Businesses by Industry Division by Statistical Area Level 2' in title:
            years.extend(re.findall(r'June (20\d{2})', title))
    wb.close()
    return sorted(set(years))

def choose_sa2(df):
    names = df['SA2 Label'].dropna().astype(str)
    if (names == 'Tarneit - North').any():
        return 'Tarneit - North'
    wyndham = names[names.str.contains('Wyndham', case=False, na=False)]
    if not wyndham.empty:
        return sorted(wyndham.unique())[0]
    return sorted(names.unique())[0]

def inspect_cabee(path, release_label, likely_sheet):
    print_file_overview(path)
    print()
    print(f'Most likely SA2-level data sheet: {likely_sheet!r}')
    print('Actual column header rows: Excel rows 5-6; pandas header=[4, 5].')
    print_raw_rows(path, likely_sheet)
    print()
    raw_loaded = load_cabee_sheet(path, likely_sheet)
    data = clean_cabee_data(raw_loaded)
    print_loaded_df_details(data, f'{release_label} / {likely_sheet}')
    print(f'Raw rows after header before dropping footer/non-SA2 rows: {len(raw_loaded):,}')
    print(f'Clean SA2 data rows after dropping footer/non-SA2 rows: {len(data):,}')
    print()
    print('SA2 code column: SA2 Code')
    print('SA2 name column: SA2 Label')
    print_sa2_samples(data, 'SA2 Label')
    print()
    employment_cols = [c for c in data.columns if c not in ['Industry Code', 'Industry Label', 'SA2 Code', 'SA2 Label']]
    print('Distinct employment-size-range categories present:')
    for col in employment_cols:
        print(f'  - {col}')
    print('\nDistinct industry division values:')
    for value in sorted(data['Industry Label'].dropna().astype(str).unique()):
        print(f'  - {value}')
    years = cabee_years_from_sheets(path)
    print('\nYear columns present: none. Years are encoded in sheet names/table titles.')
    print(f'Data years detected in table sheets: {years}')
    chosen = choose_sa2(data)
    print(f'\nGreater Melbourne SA2 selected for shape check: {chosen}')
    sample_rows = data[data['SA2 Label'].eq(chosen)].copy()
    print(f'All rows for {chosen!r} in {likely_sheet!r}:')
    print(sample_rows.to_string(index=False))
    return chosen

def inspect_erp(path, likely_sheet, chosen_sa2='Tarneit - North'):
    print_file_overview(path)
    print()
    print(f'Most likely SA2-level data sheet: {likely_sheet!r}')
    print('Actual column header row: Excel row 7; pandas header=6.')
    print_raw_rows(path, likely_sheet)
    print()
    raw_loaded = load_erp_sheet(path, likely_sheet)
    data = clean_erp_data(raw_loaded)
    print_loaded_df_details(data, f'ERP / {likely_sheet}')
    print(f'Raw rows after header before dropping footer/non-SA2 rows: {len(raw_loaded):,}')
    print(f'Clean SA2 data rows after dropping footer/non-SA2 rows: {len(data):,}')
    print()
    print('SA2 code column: SA2 code')
    print('SA2 name column: SA2 name')
    print_sa2_samples(data, 'SA2 name')
    print()
    core_cols = ['Year', 'S/T code', 'S/T name', 'GCCSA code', 'GCCSA name', 'SA4 code', 'SA4 name', 'SA3 code', 'SA3 name', 'SA2 code', 'SA2 name']
    age_cols = [c for c in data.columns if c not in core_cols]
    print('Age bands present:')
    for col in age_cols:
        print(f'  - {col}')
    years = [int(y) for y in sorted(pd.to_numeric(data['Year'], errors='coerce').dropna().astype(int).unique())]
    print(f'\nYears present: {years}')
    latest = max(years)
    print(f'\nSample for {chosen_sa2!r} in latest year {latest}, all age bands:')
    sample = data[(data['SA2 name'].eq(chosen_sa2)) & (pd.to_numeric(data['Year'], errors='coerce').eq(latest))]
    if sample.empty:
        print(f'No exact row for {chosen_sa2!r}; showing first Tarneit-like row for {latest}.')
        sample = data[(data['SA2 name'].astype(str).str.contains('Tarneit', case=False, na=False)) & (pd.to_numeric(data['Year'], errors='coerce').eq(latest))].head(1)
    if sample.empty:
        print('No comparable sample found.')
    else:
        row = sample.iloc[0]
        print(f"SA2 code: {row['SA2 code']}")
        print(f"SA2 name: {row['SA2 name']}")
        print(f"Year: {row['Year']}")
        print(pd.DataFrame({'age_band': age_cols, 'population': [row[c] for c in age_cols]}).to_string(index=False))

def extract_older_cabee_asgs_sentence(path):
    wb = load_workbook(path, read_only=True, data_only=True)
    direct_hits = []
    geography_hits = []
    for ws in wb.worksheets:
        for row_idx, row in enumerate(ws.iter_rows(values_only=True), start=1):
            text = ' '.join(str(v) for v in row if v is not None)
            if not text.strip():
                continue
            lower = text.lower()
            if 'asgs' in lower or 'australian statistical geography standard' in lower or 'edition' in lower:
                direct_hits.append((ws.title, row_idx, text))
            if 'classified to a single geography' in lower or 'smaller geographies, such as sa2 and lga' in lower:
                geography_hits.append((ws.title, row_idx, text))
    wb.close()
    print('ASGS edition sentence search in CABEE 2018-2022 workbook:')
    if direct_hits:
        for sheet, row_idx, text in direct_hits:
            print(f'  {sheet} row {row_idx}: {text}')
    else:
        print('  No explicit ASGS / Australian Statistical Geography Standard / Edition sentence found in the workbook text.')
        if geography_hits:
            sheet, row_idx, text = geography_hits[0]
            print('  Closest relevant geography methodology sentence found verbatim:')
            print(f'  {sheet} row {row_idx}: {text}')

missing = [str(path) for path in FILES.values() if not path.exists()]
if missing:
    print('Missing source file(s):')
    for path in missing:
        print(f'  - {path}')
    raise FileNotFoundError('Missing source file(s); stop inspection.')
print('Resolved source files:')
for label, path in FILES.items():
    print(f'  - {label}: {path} ({size_mb(path):.2f} MB)')


## File 1 — CABEE 2021-2025

Inspect the newest CABEE workbook first because it supplies the 2023-2024 frames. The important checks are the multi-row header location, the exact `Non employing` column label, the industry categories, and SA2 names used for later matching.

In [ ]:
try:
    SELECTED_SA2 = inspect_cabee(FILES['CABEE 2021-2025'], 'CABEE 2021-2025', 'Table 1')
except Exception:
    traceback.print_exc()


## File 2 — CABEE 2018-2022

Inspect the middle CABEE release because it supplies the 2020-2022 annualised `a` sheets. The methodology search at the end looks for geography/ASGS clues that would indicate whether a boundary correspondence step is needed.

In [ ]:
try:
    SELECTED_SA2_OLD = inspect_cabee(FILES['CABEE 2018-2022'], 'CABEE 2018-2022', '2022 a')
    print()
    extract_older_cabee_asgs_sentence(FILES['CABEE 2018-2022'])
except Exception:
    traceback.print_exc()


## File 3 — ERP by SA2 by Age and Sex 2001-2024

Inspect the ERP age/sex workbook used for the working-age denominator. The pipeline later sums ages 15-64 and uses matching ERP years through 2024 because ERP currently ends at 2024.

In [ ]:
try:
    inspect_erp(FILES['ERP by SA2 age/sex 2001-2024'], 'Table 3', globals().get('SELECTED_SA2', 'Tarneit - North'))
except Exception:
    traceback.print_exc()


## Build Analysis Table

This section builds the tidy self-employed business density analysis table and prints verification checks for review. It does not create any visualisation, GeoJSON, Plotly, or HTML outputs.

Core methodology decisions:

- Use CABEE annualised `a` sheets only for 2019-2024.
- Use `Non employing` businesses as the numerator.
- Drop CABEE `Industry Code == "X"` because those rows are currently unknown state-level dumps.
- Use working-age residents aged 15-64 as the denominator.
- Apply a working-age population floor of 500 for map inclusion.
- Keep the seven CBD/Southbank/industrial-estate SA2s in the output but flag them for the alternate view.

Outputs from this section:

- `data/processed/self_employed_business_density.csv`

### Configuration and Helper Functions

This cell defines all locked analysis-table parameters in one place: repo-relative source folders, year range, working-age bands, the 500-person denominator floor, CBD/distortion SA2 flags, output paths, expected raw-column schemas, and helper functions for loading CABEE and ERP files. It resolves the repo root from either the project folder or the `notebooks/` folder, so other developers do not need to edit absolute paths.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from openpyxl import load_workbook

CWD = Path.cwd().resolve()
if (CWD / "data").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data").exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError("Could not find the repo root containing the data/ directory.")

DATA_RAW_ABS_DIR = REPO_ROOT / "data" / "raw" / "abs"
CABEE_DIR = DATA_RAW_ABS_DIR / "cabee"
ERP_DIR = DATA_RAW_ABS_DIR / "population_age_sex"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"

YEAR_RANGE = list(range(2019, 2025))
LATEST_YEAR = max(YEAR_RANGE)
WORKING_AGE_BANDS = [
    "age_15_19", "age_20_24", "age_25_29", "age_30_34", "age_35_39",
    "age_40_44", "age_45_49", "age_50_54", "age_55_59", "age_60_64",
]
WORKING_AGE_FLOOR = 500

CBD_EXCLUSION_SA2_NAMES = [
    "Melbourne CBD - East",
    "Melbourne CBD - North",
    "Melbourne CBD - West",
    "Docklands",
    "Southbank - East",
    "Southbank (West) - South Wharf",
    "Port Melbourne Industrial",
]

CABEE_SOURCES = {
    2019: ("cabee_sa2_industry_employment_size_2017_2021.xlsx", "2019"),
    2020: ("cabee_sa2_industry_employment_size_2018_2022.xlsx", "2020 a"),
    2021: ("cabee_sa2_industry_employment_size_2018_2022.xlsx", "2021 a"),
    2022: ("cabee_sa2_industry_employment_size_2018_2022.xlsx", "2022 a"),
}
CABEE_2021_2025_FILE = "cabee_sa2_industry_employment_size_2021_2025.xlsx"
ERP_FILE = "erp_sa2_age_sex_2001_2024.xlsx"

DENSITY_CSV_OUTPUT = PROCESSED_DIR / "self_employed_business_density.csv"

CABEE_COLUMNS = [
    "Industry Code", "Industry Label", "SA2 Code", "SA2 Label",
    "Non employing", "1-4 Employees", "5-19 Employees",
    "20-199 Employees", "200+ Employees", "Total",
]
ERP_COLUMNS = [
    "year", "st_code", "st_name",
    "gccsa_code", "gccsa_name",
    "sa4_code", "sa4_name",
    "sa3_code", "sa3_name",
    "sa2_code", "sa2_name",
    "age_0_4", "age_5_9", "age_10_14", "age_15_19", "age_20_24",
    "age_25_29", "age_30_34", "age_35_39", "age_40_44", "age_45_49",
    "age_50_54", "age_55_59", "age_60_64", "age_65_69", "age_70_74",
    "age_75_79", "age_80_84", "age_85_plus", "total_persons",
]
FINAL_COLUMNS = [
    "year", "sa2_code", "sa2_name", "non_employing_count", "working_age_pop",
    "total_persons", "non_employing_per_1000_working_age", "change_abs_since_2019",
    "change_pct_since_2019", "is_cbd_excluded", "has_sufficient_population",
    "include_in_primary_view", "include_in_alternate_view",
]


def resolve_sheet_name(workbook_path, requested_sheet):
    wb = load_workbook(workbook_path, read_only=True, data_only=True)
    try:
        stripped = {sheet.strip(): sheet for sheet in wb.sheetnames}
    finally:
        wb.close()
    requested_key = str(requested_sheet).strip()
    if requested_key in stripped:
        return stripped[requested_key]
    available = ", ".join(sorted(stripped))
    raise ValueError(f"Sheet {requested_sheet!r} not found in {workbook_path.name}; available sheets: {available}")


def parse_cabee_2021_2025_mapping(workbook_path):
    resolved = {}
    wb = load_workbook(workbook_path, read_only=True, data_only=True)
    try:
        stripped = {sheet.strip(): sheet for sheet in wb.sheetnames}
        for requested in ["Table 1", "Table 2", "Table 3"]:
            actual = stripped.get(requested)
            if actual is None:
                raise ValueError(f"Missing expected sheet {requested!r} in {workbook_path.name}")
            title = " ".join(str(v) for v in next(wb[actual].iter_rows(min_row=4, max_row=4, values_only=True)) if v is not None)
            match = re.search(r"June (20\d{2})", title)
            if not match:
                raise ValueError(f"Could not parse year from {workbook_path.name}/{actual} title row: {title!r}")
            year = int(match.group(1))
            resolved[year] = actual
    finally:
        wb.close()
    expected = {2023, 2024, 2025}
    if set(resolved) != expected:
        raise ValueError(f"2021-2025 CABEE sheet parsing expected {sorted(expected)}, got {sorted(resolved)}")
    return resolved


def normalize_sa2_code(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype("string")


def load_cabee_sheet(file_name, sheet_name, year, *, emit=print):
    workbook_path = CABEE_DIR / file_name
    actual_sheet = resolve_sheet_name(workbook_path, sheet_name)
    raw = pd.read_excel(
        workbook_path,
        sheet_name=actual_sheet,
        header=[4, 5],
        engine="openpyxl",
    )
    raw = raw.iloc[:, :len(CABEE_COLUMNS)].copy()
    raw.columns = CABEE_COLUMNS
    raw = raw[pd.to_numeric(raw["SA2 Code"], errors="coerce").notna()].copy()
    raw["SA2 Code"] = normalize_sa2_code(raw["SA2 Code"])
    raw["Industry Code"] = raw["Industry Code"].astype("string")
    raw["Industry Label"] = raw["Industry Label"].astype("string")
    raw["SA2 Label"] = raw["SA2 Label"].astype("string")
    raw["Non employing"] = pd.to_numeric(raw["Non employing"], errors="coerce").astype("Int64")
    raw["year"] = int(year)
    clean = raw[["year", "SA2 Code", "SA2 Label", "Industry Code", "Industry Label", "Non employing"]].copy()
    emit(f"Loaded {year} from {file_name}/{actual_sheet}: {len(clean):,} rows, {clean['SA2 Code'].nunique():,} unique SA2s")
    return clean


def aggregate_non_employing(cabee_long):
    filtered = cabee_long[cabee_long["Industry Code"].ne("X")].copy()
    return (
        filtered.groupby(["year", "SA2 Code", "SA2 Label"], as_index=False)["Non employing"]
        .sum()
        .rename(columns={
            "SA2 Code": "sa2_code",
            "SA2 Label": "sa2_name",
            "Non employing": "non_employing_count",
        })
    )


def load_erp_working_age():
    erp_path = ERP_DIR / ERP_FILE
    erp = pd.read_excel(erp_path, sheet_name="Table 3", header=None, skiprows=7, engine="openpyxl")
    if erp.shape[1] != len(ERP_COLUMNS):
        raise ValueError(f"ERP Table 3 expected exactly {len(ERP_COLUMNS)} columns, got {erp.shape[1]}")
    erp.columns = ERP_COLUMNS
    erp = erp[pd.to_numeric(erp["sa2_code"], errors="coerce").notna()].copy()
    erp["year"] = pd.to_numeric(erp["year"], errors="coerce").astype("Int64")
    erp = erp[erp["year"].notna()].copy()
    erp["year"] = erp["year"].astype(int)
    erp["sa2_code"] = normalize_sa2_code(erp["sa2_code"])
    erp["sa2_name"] = erp["sa2_name"].astype("string")
    for col in [c for c in erp.columns if c.startswith("age_")] + ["total_persons"]:
        erp[col] = pd.to_numeric(erp[col], errors="coerce").astype("Int64")
    erp["working_age_pop"] = erp[WORKING_AGE_BANDS].sum(axis=1).astype("Int64")
    return erp


def format_df(frame, columns=None, max_rows=None):
    out = frame.copy()
    if columns is not None:
        out = out[columns]
    if max_rows is not None:
        out = out.head(max_rows)
    return out.to_string(index=False)

### Run Pipeline, Verification, and Write Outputs

This cell executes the full analysis-table build. It validates source files, resolves CABEE sheet names, loads all seven years, checks boundary stability, compares overlapping CABEE releases, aggregates non-employing businesses, joins the ERP denominator, computes per-1,000 and change-since-2019 metrics, applies view flags, writes the analysis CSV. The printed sections are intentionally verbose because they are the audit trail for stakeholder review.

In [ ]:
def say(message=""):
    print(str(message))


def print_section(title):
    say()
    say(f"=== {title} ===")


def abort_if_missing_sources():
    required = [
        CABEE_DIR / "cabee_sa2_industry_employment_size_2017_2021.xlsx",
        CABEE_DIR / "cabee_sa2_industry_employment_size_2018_2022.xlsx",
        CABEE_DIR / CABEE_2021_2025_FILE,
        ERP_DIR / ERP_FILE,
    ]
    missing = [path for path in required if not path.exists()]
    if missing:
        say("Missing source file(s):")
        for path in missing:
            say(f"  - {path}")
        raise FileNotFoundError("Missing source file(s); aborting self-employed business density pipeline")


PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
abort_if_missing_sources()

print_section("Source files")
for path in [
    CABEE_DIR / "cabee_sa2_industry_employment_size_2017_2021.xlsx",
    CABEE_DIR / "cabee_sa2_industry_employment_size_2018_2022.xlsx",
    CABEE_DIR / CABEE_2021_2025_FILE,
    ERP_DIR / ERP_FILE,
]:
    say(f"{path} ({path.stat().st_size / (1024 * 1024):.2f} MB)")

print_section("Resolved CABEE 2021-2025 annualised sheet mapping")
cabee_2021_2025_mapping = parse_cabee_2021_2025_mapping(CABEE_DIR / CABEE_2021_2025_FILE)
for year in sorted(cabee_2021_2025_mapping):
    say(f"{year}: {CABEE_2021_2025_FILE}/{cabee_2021_2025_mapping[year]}")

all_cabee_sources = dict(CABEE_SOURCES)
for year, sheet in cabee_2021_2025_mapping.items():
    if year in YEAR_RANGE:
        all_cabee_sources[year] = (CABEE_2021_2025_FILE, sheet)
if sorted(all_cabee_sources) != YEAR_RANGE:
    raise ValueError(f"Expected CABEE sources for {YEAR_RANGE}, got {sorted(all_cabee_sources)}")

print_section("CABEE load")
cabee_frames = []
for year in YEAR_RANGE:
    file_name, sheet_name = all_cabee_sources[year]
    frame = load_cabee_sheet(file_name, sheet_name, year, emit=say)
    if frame.empty:
        raise ValueError(f"CABEE load for {year} produced zero rows; aborting")
    cabee_frames.append(frame)
cabee_long = pd.concat(cabee_frames, ignore_index=True)

print_section("Boundary stability check")
sa2_sets_by_year = {
    year: set(cabee_long.loc[cabee_long["year"].eq(year), "SA2 Code"].dropna().astype(str))
    for year in YEAR_RANGE
}
boundary_summary = pd.DataFrame(
    {"year": year, "unique_sa2_count": len(sa2_sets_by_year[year])}
    for year in YEAR_RANGE
)
say(boundary_summary.to_string(index=False))
sym_diff = sa2_sets_by_year[2019].symmetric_difference(sa2_sets_by_year[LATEST_YEAR])
say(f"Symmetric difference between 2019 and latest-year SA2 code sets: {len(sym_diff):,}")
if len(sym_diff) > 50:
    raise ValueError("boundary correspondence required, halting before downstream steps")

print_section("Overlap check for 2017-2021 vs 2018-2022 releases")
for overlap_year in [2020, 2021]:
    old = load_cabee_sheet(
        "cabee_sa2_industry_employment_size_2017_2021.xlsx",
        f"{overlap_year} a",
        overlap_year,
        emit=lambda _msg: None,
    )
    new = load_cabee_sheet(
        "cabee_sa2_industry_employment_size_2018_2022.xlsx",
        f"{overlap_year} a",
        overlap_year,
        emit=lambda _msg: None,
    )
    old_agg = aggregate_non_employing(old).rename(columns={"non_employing_count": "old_non_employing"})
    new_agg = aggregate_non_employing(new).rename(columns={"non_employing_count": "new_non_employing"})
    qa = old_agg[["sa2_code", "old_non_employing"]].merge(
        new_agg[["sa2_code", "new_non_employing"]],
        on="sa2_code",
        how="inner",
    )
    denominator = qa["new_non_employing"].replace(0, pd.NA)
    qa["abs_relative_diff"] = (qa["old_non_employing"] - qa["new_non_employing"]).abs() / denominator
    qa.loc[qa["new_non_employing"].eq(0) & qa["old_non_employing"].eq(0), "abs_relative_diff"] = 0
    median_abs_rel_diff = qa["abs_relative_diff"].replace([np.inf, -np.inf], pd.NA).dropna().median()
    outlier_count = int((qa["abs_relative_diff"] > 0.10).sum())
    say(f"{overlap_year}: median absolute relative difference = {median_abs_rel_diff:.2%}; SA2s >10% = {outlier_count:,} of {len(qa):,}")

print_section("Industry filter and aggregation")
cabee_filtered = cabee_long[cabee_long["Industry Code"].ne("X")].copy()
say(f"CABEE long rows before dropping Industry Code X: {len(cabee_long):,}")
say(f"CABEE long rows after dropping Industry Code X: {len(cabee_filtered):,}")
cabee_aggregated = aggregate_non_employing(cabee_long)
say(f"Aggregated CABEE rows: {len(cabee_aggregated):,}")
say(cabee_aggregated.groupby("year").size().rename("rows").to_string())
if (cabee_aggregated.groupby("year").size() == 0).any():
    raise ValueError("At least one year has zero CABEE rows after filtering; aborting")

print_section("ERP load and Greater Melbourne filter")
erp = load_erp_working_age()
greater_melbourne_erp = erp[erp["gccsa_name"].eq("Greater Melbourne")].copy()
greater_melbourne_sa2_codes = set(greater_melbourne_erp["sa2_code"].dropna().astype(str))
say(f"Resolved Greater Melbourne SA2 count: {len(greater_melbourne_sa2_codes):,}")
say(f"ERP working-age denominators use matching ERP years {YEAR_RANGE[0]}-{LATEST_YEAR}")

erp_for_join = greater_melbourne_erp[greater_melbourne_erp["year"].isin(YEAR_RANGE)].copy()
erp_for_join = erp_for_join[["year", "sa2_code", "sa2_name", "working_age_pop", "total_persons"]].copy()
erp_for_join["sa2_code"] = erp_for_join["sa2_code"].astype("string")

erp_working_age = erp_for_join.copy()

cabee_aggregated = cabee_aggregated[cabee_aggregated["sa2_code"].astype(str).isin(greater_melbourne_sa2_codes)].copy()
cabee_aggregated["sa2_code"] = cabee_aggregated["sa2_code"].astype("string")
say(f"CABEE aggregated Greater Melbourne rows: {len(cabee_aggregated):,}")
say(cabee_aggregated.groupby("year").size().rename("rows").to_string())

print_section("Join and metric computation")
df = erp_for_join.merge(
    cabee_aggregated,
    on=["year", "sa2_code"],
    how="left",
    suffixes=("_erp", "_cabee"),
)
df["sa2_name"] = df["sa2_name_cabee"].combine_first(df["sa2_name_erp"])
df = df.drop(columns=["sa2_name_erp", "sa2_name_cabee"])
df["non_employing_count"] = df["non_employing_count"].astype("Int64")
df["working_age_pop"] = df["working_age_pop"].astype("Int64")
df["total_persons"] = df["total_persons"].astype("Int64")
df["non_employing_per_1000_working_age"] = np.where(
    df["working_age_pop"] > 0,
    (df["non_employing_count"] / df["working_age_pop"] * 1000).round(1),
    np.nan,
)
zero_denominator_rows = df["working_age_pop"].le(0).sum()
say(f"Rows with working_age_pop <= 0 set to NaN for per-1000/change metrics: {int(zero_denominator_rows):,}")

baseline = (
    df[df["year"].eq(2019)][["sa2_code", "non_employing_per_1000_working_age"]]
    .rename(columns={"non_employing_per_1000_working_age": "baseline_2019"})
)
df = df.merge(baseline, on="sa2_code", how="left")
missing_or_invalid_baseline = df.groupby("sa2_code")["baseline_2019"].first().isna()
say(f"SA2s without valid 2019 baseline metric: {int(missing_or_invalid_baseline.sum()):,}")
df["change_abs_since_2019"] = (
    df["non_employing_per_1000_working_age"] - df["baseline_2019"]
).round(1)
df["change_pct_since_2019"] = np.where(
    df["baseline_2019"] > 0,
    ((df["non_employing_per_1000_working_age"] / df["baseline_2019"] - 1) * 100).round(1),
    np.nan,
)

df["is_cbd_excluded"] = df["sa2_name"].isin(CBD_EXCLUSION_SA2_NAMES)
df["has_sufficient_population"] = df["working_age_pop"] >= WORKING_AGE_FLOOR
df["include_in_primary_view"] = (~df["is_cbd_excluded"]) & df["has_sufficient_population"]
df["include_in_alternate_view"] = df["has_sufficient_population"]

final_df = df[FINAL_COLUMNS].sort_values(["year", "sa2_name"]).reset_index(drop=True)

print_section("1. Total row count of final table")
say(f"Total rows: {len(final_df):,}")

print_section("2. Count of rows per year where include_in_primary_view is True")
say(final_df[final_df["include_in_primary_view"]].groupby("year").size().rename("primary_view_rows").to_string())

print_section("3. CBD-excluded SA2 names found")
cbd_found = sorted(final_df.loc[final_df["is_cbd_excluded"], "sa2_name"].dropna().unique())
for name in cbd_found:
    say(f"- {name}")
cbd_missing = sorted(set(CBD_EXCLUSION_SA2_NAMES) - set(cbd_found))
if cbd_missing:
    say("Missing CBD exclusion names:")
    for name in cbd_missing:
        say(f"- {name}")
if len(cbd_found) < 6:
    say(f"WARNING: only {len(cbd_found)} of 7 CBD exclusion SA2s found; continuing")

print_section("4. SA2/year rows below the working-age floor")
low_population = final_df[~final_df["has_sufficient_population"]].sort_values(["working_age_pop", "year", "sa2_name"])
say(format_df(low_population[["year", "sa2_name", "working_age_pop", "non_employing_count", "non_employing_per_1000_working_age"]]))

frame_latest_primary = final_df[(final_df["year"].eq(LATEST_YEAR)) & (final_df["include_in_primary_view"])].copy()

metric_cols = [
    "sa2_name", "working_age_pop", "non_employing_count",
    "non_employing_per_1000_working_age", "change_abs_since_2019", "change_pct_since_2019",
]

print_section(f"5. Top 15 SA2s by non_employing_per_1000_working_age, {LATEST_YEAR} primary view")
say(format_df(frame_latest_primary.nlargest(15, "non_employing_per_1000_working_age"), metric_cols))

print_section(f"6. Bottom 15 SA2s by non_employing_per_1000_working_age, {LATEST_YEAR} primary view")
say(format_df(frame_latest_primary.nsmallest(15, "non_employing_per_1000_working_age"), metric_cols))

print_section(f"7. Top 15 SA2s by change_abs_since_2019, {LATEST_YEAR} primary view")
say(format_df(frame_latest_primary.nlargest(15, "change_abs_since_2019"), metric_cols))

print_section(f"8. Top 15 SA2s by change_pct_since_2019, {LATEST_YEAR} primary view, baseline >=10 per 1000")
frame_latest_with_baseline = df[(df["year"].eq(LATEST_YEAR)) & (df["include_in_primary_view"]) & (df["baseline_2019"] >= 10)].copy()
say(format_df(frame_latest_with_baseline.nlargest(15, "change_pct_since_2019"), metric_cols + ["baseline_2019"]))

print_section("9. Tarneit - North all-year anchor")
tarneit = final_df[final_df["sa2_name"].eq("Tarneit - North")].sort_values("year")
say(format_df(tarneit[["year", "non_employing_count", "working_age_pop", "non_employing_per_1000_working_age", "change_abs_since_2019"]]))

print_section(f"10. Growth-corridor SA2 2019 vs {LATEST_YEAR} comparison")
growth_requested = ["Tarneit - North", "Cranbourne East - North", "Mickleham - Yuroke"]
for name in growth_requested:
    if name in set(final_df["sa2_name"]):
        selected_name = name
    else:
        matches = sorted([candidate for candidate in final_df["sa2_name"].dropna().unique() if name.split(" - ")[0] in candidate])
        selected_name = matches[0] if matches else None
    if selected_name is None:
        say(f"No match found for {name}")
        continue
    comparison = final_df[(final_df["sa2_name"].eq(selected_name)) & (final_df["year"].isin([2019, LATEST_YEAR]))]
    say(f"Requested: {name}; using: {selected_name}")
    say(format_df(comparison[["year", "sa2_name", "non_employing_count", "working_age_pop", "non_employing_per_1000_working_age", "change_abs_since_2019", "change_pct_since_2019"]]))

print_section(f"11. Established middle-ring SA2 2019 vs {LATEST_YEAR} comparison")
established_requested = ["Brighton (Vic.)", "Box Hill", "Camberwell"]
for name in established_requested:
    if name in set(final_df["sa2_name"]):
        selected_name = name
    else:
        matches = sorted([candidate for candidate in final_df["sa2_name"].dropna().unique() if name in candidate])
        selected_name = matches[0] if matches else None
    if selected_name is None:
        say(f"No match found for {name}")
        continue
    comparison = final_df[(final_df["sa2_name"].eq(selected_name)) & (final_df["year"].isin([2019, LATEST_YEAR]))]
    say(f"Requested: {name}; using: {selected_name}")
    say(format_df(comparison[["year", "sa2_name", "non_employing_count", "working_age_pop", "non_employing_per_1000_working_age", "change_abs_since_2019", "change_pct_since_2019"]]))

print_section(f"12. {LATEST_YEAR} primary-view distribution summary")
summary = frame_latest_primary["non_employing_per_1000_working_age"].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
summary = summary.rename(index={"10%": "p10", "25%": "p25", "50%": "p50", "75%": "p75", "90%": "p90", "95%": "p95"})
say(summary.loc[["min", "p10", "p25", "p50", "p75", "p90", "p95", "max"]].to_string())

final_df.to_csv(DENSITY_CSV_OUTPUT, index=False)

print_section("Output verification")
say(f"Wrote CSV: {DENSITY_CSV_OUTPUT} ({DENSITY_CSV_OUTPUT.stat().st_size:,} bytes)")
read_back = pd.read_csv(DENSITY_CSV_OUTPUT)
say(f"CSV read-back rows: {len(read_back):,}; columns: {list(read_back.columns)}")
say("self-employed business density pipeline completed successfully.")


## Industry Filter And Final Map Build

The cells below build the canonical deliverable map: the industry-filtered interactive HTML, Story Mode walkthrough, footer, and static PNG preview. The older aggregate-only output path has been retired to keep the notebook focused for future maintainers.


### Build Industry-Level Analysis Table

This cell reruns the CABEE loading logic with the ANZSIC division dimension preserved. It adds 19 industry rows plus an `ALL` aggregate row for every `(year, SA2)` in the existing self-employed business density analysis table. The `ALL` rows are copied from `self_employed_business_density.csv` and must match exactly.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from openpyxl import load_workbook

CWD = Path.cwd().resolve()
if (CWD / "data").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "data").exists():
    REPO_ROOT = CWD.parent
else:
    raise FileNotFoundError("Could not find the repo root containing the data/ directory.")

CABEE_DIR = REPO_ROOT / "data" / "raw" / "abs" / "cabee"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
DENSITY_CSV_PATH = PROCESSED_DIR / "self_employed_business_density.csv"
BY_INDUSTRY_OUTPUT = PROCESSED_DIR / "self_employed_business_density_by_industry.csv"

YEAR_RANGE = list(range(2019, 2025))
LATEST_YEAR = max(YEAR_RANGE)
CABEE_2017_2021 = "cabee_sa2_industry_employment_size_2017_2021.xlsx"
CABEE_2018_2022 = "cabee_sa2_industry_employment_size_2018_2022.xlsx"
CABEE_2021_2025 = "cabee_sa2_industry_employment_size_2021_2025.xlsx"
CABEE_SOURCES = {
    2019: (CABEE_2017_2021, "2019"),
    2020: (CABEE_2018_2022, "2020 a"),
    2021: (CABEE_2018_2022, "2021 a"),
    2022: (CABEE_2018_2022, "2022 a"),
}
CABEE_COLUMNS = [
    "Industry Code", "Industry Label", "SA2 Code", "SA2 Label",
    "Non employing", "1-4 Employees", "5-19 Employees",
    "20-199 Employees", "200+ Employees", "Total",
]
ANZSIC_DIVISIONS = {
    "A": "Agriculture, Forestry and Fishing",
    "B": "Mining",
    "C": "Manufacturing",
    "D": "Electricity, Gas, Water and Waste Services",
    "E": "Construction",
    "F": "Wholesale Trade",
    "G": "Retail Trade",
    "H": "Accommodation and Food Services",
    "I": "Transport, Postal and Warehousing",
    "J": "Information Media and Telecommunications",
    "K": "Financial and Insurance Services",
    "L": "Rental, Hiring and Real Estate Services",
    "M": "Professional, Scientific and Technical Services",
    "N": "Administrative and Support Services",
    "O": "Public Administration and Safety",
    "P": "Education and Training",
    "Q": "Health Care and Social Assistance",
    "R": "Arts and Recreation Services",
    "S": "Other Services",
}
CBD_EXCLUSION_SA2_NAMES = [
    "Melbourne CBD - East", "Melbourne CBD - North", "Melbourne CBD - West",
    "Docklands", "Southbank - East", "Southbank (West) - South Wharf",
    "Port Melbourne Industrial",
]
FINAL_COLUMNS_BY_INDUSTRY = [
    "year", "sa2_code", "sa2_name", "industry_division", "non_employing_count",
    "working_age_pop", "total_persons", "non_employing_per_1000_working_age",
    "change_abs_since_2019", "change_pct_since_2019", "is_cbd_excluded",
    "has_sufficient_population", "include_in_primary_view", "include_in_alternate_view",
]


def normalize_sa2_code(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.astype("Int64").astype("string")


def resolve_sheet_name(workbook_path, requested_sheet):
    wb = load_workbook(workbook_path, read_only=True, data_only=True)
    try:
        stripped = {sheet.strip(): sheet for sheet in wb.sheetnames}
    finally:
        wb.close()
    requested_key = str(requested_sheet).strip()
    if requested_key in stripped:
        return stripped[requested_key]
    raise ValueError(f"Sheet {requested_sheet!r} not found in {workbook_path.name}")


def parse_cabee_2021_2025_mapping(workbook_path):
    resolved = {}
    wb = load_workbook(workbook_path, read_only=True, data_only=True)
    try:
        stripped = {sheet.strip(): sheet for sheet in wb.sheetnames}
        for requested in ["Table 1", "Table 2", "Table 3"]:
            actual = stripped.get(requested)
            if actual is None:
                raise ValueError(f"Missing expected sheet {requested!r} in {workbook_path.name}")
            title = " ".join(
                str(value)
                for value in next(wb[actual].iter_rows(min_row=4, max_row=4, values_only=True))
                if value is not None
            )
            match = re.search(r"June (20\d{2})", title)
            if not match:
                raise ValueError(f"Could not parse year from {workbook_path.name}/{actual}: {title!r}")
            resolved[int(match.group(1))] = actual
    finally:
        wb.close()
    expected = {2023, 2024, 2025}
    if set(resolved) != expected:
        raise ValueError(f"2021-2025 CABEE sheet parsing expected {sorted(expected)}, got {sorted(resolved)}")
    return resolved


def load_cabee_industry_sheet(file_name, sheet_name, year):
    workbook_path = CABEE_DIR / file_name
    actual_sheet = resolve_sheet_name(workbook_path, sheet_name)
    raw = pd.read_excel(workbook_path, sheet_name=actual_sheet, header=[4, 5], engine="openpyxl")
    raw = raw.iloc[:, :len(CABEE_COLUMNS)].copy()
    raw.columns = CABEE_COLUMNS
    raw = raw[pd.to_numeric(raw["SA2 Code"], errors="coerce").notna()].copy()
    raw["year"] = int(year)
    raw["sa2_code"] = normalize_sa2_code(raw["SA2 Code"])
    raw["sa2_name"] = raw["SA2 Label"].astype("string")
    raw["industry_division"] = raw["Industry Code"].astype("string")
    raw["non_employing_count"] = pd.to_numeric(raw["Non employing"], errors="coerce").fillna(0).astype(int)
    clean = raw[["year", "sa2_code", "sa2_name", "industry_division", "non_employing_count"]].copy()
    clean = clean[clean["industry_division"].ne("X")].copy()
    clean = clean[clean["industry_division"].isin(ANZSIC_DIVISIONS)].copy()
    return clean


for path in [CABEE_DIR / CABEE_2017_2021, CABEE_DIR / CABEE_2018_2022, CABEE_DIR / CABEE_2021_2025, DENSITY_CSV_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {path}")

cabee_2021_2025_mapping = parse_cabee_2021_2025_mapping(CABEE_DIR / CABEE_2021_2025)
all_sources = dict(CABEE_SOURCES)
for year, sheet in cabee_2021_2025_mapping.items():
    if year in YEAR_RANGE:
        all_sources[year] = (CABEE_2021_2025, sheet)
if sorted(all_sources) != YEAR_RANGE:
    raise ValueError(f"Expected CABEE sources for {YEAR_RANGE}, got {sorted(all_sources)}")

density_base = pd.read_csv(DENSITY_CSV_PATH, dtype={"sa2_code": str})
density_base["sa2_code"] = density_base["sa2_code"].astype("string")
base_keys = density_base[[
    "year", "sa2_code", "sa2_name", "working_age_pop", "total_persons",
    "is_cbd_excluded", "has_sufficient_population", "include_in_primary_view",
    "include_in_alternate_view",
]].copy()

cabee_industry_long = pd.concat(
    [load_cabee_industry_sheet(*all_sources[year], year) for year in YEAR_RANGE],
    ignore_index=True,
)
industry_counts = (
    cabee_industry_long
    .groupby(["year", "sa2_code", "industry_division"], as_index=False)["non_employing_count"]
    .sum()
)
industry_counts["sa2_code"] = industry_counts["sa2_code"].astype("string")

industry_frames = []
for division in ANZSIC_DIVISIONS:
    frame = base_keys.copy()
    frame["industry_division"] = division
    frame = frame.merge(
        industry_counts[industry_counts["industry_division"].eq(division)][["year", "sa2_code", "non_employing_count"]],
        on=["year", "sa2_code"],
        how="left",
    )
    frame["non_employing_count"] = frame["non_employing_count"].fillna(0).astype(int)
    frame["non_employing_per_1000_working_age"] = np.where(
        frame["working_age_pop"] > 0,
        (frame["non_employing_count"] / frame["working_age_pop"] * 1000).round(1),
        np.nan,
    )
    baseline = (
        frame[frame["year"].eq(2019)][["sa2_code", "non_employing_per_1000_working_age"]]
        .rename(columns={"non_employing_per_1000_working_age": "baseline_2019"})
    )
    frame = frame.merge(baseline, on="sa2_code", how="left")
    frame["change_abs_since_2019"] = (frame["non_employing_per_1000_working_age"] - frame["baseline_2019"]).round(1)
    frame["change_pct_since_2019"] = np.where(
        frame["baseline_2019"] > 0,
        ((frame["non_employing_per_1000_working_age"] / frame["baseline_2019"] - 1) * 100).round(1),
        np.nan,
    )
    frame = frame.drop(columns=["baseline_2019"])
    industry_frames.append(frame[FINAL_COLUMNS_BY_INDUSTRY])

all_frame = density_base.copy()
all_frame.insert(3, "industry_division", "ALL")
all_frame = all_frame[FINAL_COLUMNS_BY_INDUSTRY]
by_industry = pd.concat([all_frame] + industry_frames, ignore_index=True)
by_industry = by_industry.sort_values(["industry_division", "year", "sa2_name"]).reset_index(drop=True)

# Required verification checks before writing output.
SUM_CHECK_RESULTS = {}
for check_year in [2019, 2022, LATEST_YEAR]:
    industry_sum = (
        by_industry[
            by_industry["year"].eq(check_year)
            & by_industry["include_in_primary_view"]
            & by_industry["industry_division"].isin(ANZSIC_DIVISIONS)
        ]
        .groupby(["year", "sa2_code"], as_index=False)["non_employing_count"]
        .sum()
        .rename(columns={"non_employing_count": "industry_sum"})
    )
    base_compare = density_base[
        density_base["year"].eq(check_year) & density_base["include_in_primary_view"]
    ][["year", "sa2_code", "non_employing_count"]].copy()
    compare = base_compare.merge(industry_sum, on=["year", "sa2_code"], how="left")
    compare["industry_sum"] = compare["industry_sum"].fillna(0).astype(int)
    SUM_CHECK_RESULTS[check_year] = bool(compare["non_employing_count"].astype(int).eq(compare["industry_sum"].astype(int)).all())
    if not SUM_CHECK_RESULTS[check_year]:
        mismatches = compare[~compare["non_employing_count"].astype(int).eq(compare["industry_sum"].astype(int))]
        raise ValueError(f"Industry sum check failed for {check_year}: {len(mismatches)} mismatched SA2s")

all_check_cols = [col for col in density_base.columns]
all_compare = by_industry[by_industry["industry_division"].eq("ALL")].drop(columns=["industry_division"]).reset_index(drop=True)
base_compare = density_base.sort_values(["year", "sa2_name"]).reset_index(drop=True)
ALL_CHECK_RESULT = bool(all_compare[all_check_cols].equals(base_compare[all_check_cols]))
if not ALL_CHECK_RESULT:
    raise ValueError("Aggregate ALL pseudo-division does not match self_employed_business_density.csv exactly")

BY_INDUSTRY_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
by_industry.to_csv(BY_INDUSTRY_OUTPUT, index=False)

print(f"Wrote {BY_INDUSTRY_OUTPUT} ({BY_INDUSTRY_OUTPUT.stat().st_size:,} bytes)")
print("Industry sum checks, primary-view SA2s:")
for year, passed in SUM_CHECK_RESULTS.items():
    print(f"  {year}: {'PASS' if passed else 'FAIL'}")
print(f"ALL pseudo-division exact match: {'PASS' if ALL_CHECK_RESULT else 'FAIL'}")
print(f"Rows written: {len(by_industry):,}")


### Build Industry-Filtered HTML

This cell writes a new self-employed-economy HTML map. It embeds one compact JavaScript data payload for all industries, but renders only one trace per view at a time. The industry dropdown, year slider, view toggle, and Play/Pause controls all update the map with `Plotly.react()`.

In [ ]:
from pathlib import Path
import json
import re
from html import escape as html_escape

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.offline import get_plotlyjs

GPKG_PATH = REPO_ROOT / "data" / "processed" / "melbourne.gpkg"
OUT_DIR = REPO_ROOT / "outputs" / "maps"
HTML_OUT = OUT_DIR / "self_employed_business_density.html"
OUT_DIR.mkdir(parents=True, exist_ok=True)

by_industry = pd.read_csv(BY_INDUSTRY_OUTPUT, dtype={"sa2_code": str})
by_industry["sa2_code"] = by_industry["sa2_code"].astype(str)

sa2_gdf = gpd.read_file(GPKG_PATH, layer="greater_melb_sa2")
sa2_gdf = sa2_gdf[["SA2_CODE_2021", "SA2_NAME_2021", "geometry"]].copy()
sa2_gdf["SA2_CODE_2021"] = sa2_gdf["SA2_CODE_2021"].astype(str)
sa2_gdf = sa2_gdf.to_crs(epsg=4326)
sa2_gdf["geometry"] = sa2_gdf.geometry.simplify(1e-4, preserve_topology=True)

lga = gpd.read_file(GPKG_PATH, layer="greater_melb_local_gov_2025_any_intersect")
lga = lga[["LGA_NAME_2025", "geometry"]].to_crs(sa2_gdf.crs)
sa2_centroids = sa2_gdf.to_crs(7855).copy()
sa2_centroids["geometry"] = sa2_centroids.geometry.centroid
sa2_centroids = sa2_centroids.to_crs(sa2_gdf.crs)
joined = gpd.sjoin(
    sa2_centroids[["SA2_CODE_2021", "geometry"]],
    lga,
    how="left",
    predicate="within",
)
sa2_council = joined[["SA2_CODE_2021", "LGA_NAME_2025"]].rename(columns={"LGA_NAME_2025": "council"})
rows_before = len(by_industry)
by_industry = by_industry.merge(sa2_council, left_on="sa2_code", right_on="SA2_CODE_2021", how="left").drop(columns=["SA2_CODE_2021"])
assert len(by_industry) == rows_before, "Row count changed during council merge"
by_industry["council"] = by_industry["council"].fillna("Unknown")

sa2_geojson = json.loads(sa2_gdf[["SA2_CODE_2021", "geometry"]].to_json())
YEARS = YEAR_RANGE
VIEWS = {
    "residential": {"label": "Residential Melbourne", "rate_view": "Residential", "include_col": "include_in_primary_view"},
    "all_melb": {"label": "All Melbourne (incl. CBD)", "rate_view": "All", "include_col": "include_in_alternate_view"},
}
INDUSTRY_LABELS = {"ALL": "All self-employed (aggregate)"} | {code: f"{code} - {name}" for code, name in ANZSIC_DIVISIONS.items()}

# Industry order: aggregate pinned first, then divisions by 2024 primary-view total descending.
industry_totals_latest = (
    by_industry[
        by_industry["year"].eq(LATEST_YEAR)
        & by_industry["include_in_primary_view"]
        & by_industry["industry_division"].isin(ANZSIC_DIVISIONS)
    ]
    .groupby("industry_division")["non_employing_count"]
    .sum()
    .sort_values(ascending=False)
)
INDUSTRY_ORDER = ["ALL"] + industry_totals_latest.index.tolist()


def tooltip_industry_label(industry):
    if industry == "ALL":
        return "All industries"
    return re.sub(r"\band\b", "&", ANZSIC_DIVISIONS[industry])


def clean_number(value):
    if pd.isna(value):
        return None
    return float(value)


def clean_int(value):
    if pd.isna(value):
        return None
    return int(round(float(value)))


def format_density(value):
    if value is None or pd.isna(value):
        return "n/a"
    value = float(value)
    if value == 0:
        return "0"
    if abs(value) >= 10:
        return f"{value:.0f}"
    if abs(value) >= 1:
        return f"{value:.1f}"
    return f"{value:.2f}"


def format_trend_delta_abs(change_abs):
    """Magnitude for trend bracket; prefer integers when |Δ| >= 1 for readability."""
    if change_abs is None or pd.isna(change_abs):
        return "0"
    a = abs(float(change_abs))
    if a >= 1:
        return str(int(round(a)))
    if a == 0:
        return "0"
    # Preserve sub-1 precision (e.g. 0.45) without trailing noise
    s = f"{a:.2f}".rstrip("0").rstrip(".")
    return s or "0"


def format_trend_pct(value):
    """Keep large changes tidy, but do not round small non-zero changes to 0%."""
    if value is None or pd.isna(value):
        return "n/a"
    a = abs(float(value))
    if a == 0:
        return "0%"
    if a < 10:
        text = f"{a:.1f}".rstrip("0").rstrip(".")
    else:
        text = f"{a:.0f}"
    return f"{text}%"


def multiplier_for(density, melbourne_rate):
    if density is None or pd.isna(density):
        return np.nan
    if pd.isna(melbourne_rate) or melbourne_rate == 0:
        return np.nan
    return float(density) / float(melbourne_rate)


def multiplier_label_for(multiplier):
    if pd.isna(multiplier):
        return "Comparison not available"
    if multiplier == 0:
        return "No local businesses in this category"
    if 0.95 <= multiplier <= 1.05:
        return "Around the Greater Melbourne average"
    if multiplier >= 1.05:
        return f"{multiplier:.1f}× Greater Melbourne average"
    return f"{multiplier:.1f}× Greater Melbourne average (below)"


def trend_arrow_html_for(year, change_abs, trend_pct):
    if int(year) == 2019:
        return ""
    if pd.isna(change_abs) or pd.isna(trend_pct):
        return ""
    change_abs = float(change_abs)
    trend_pct = float(trend_pct)
    abs_str = format_trend_delta_abs(change_abs)
    sign = "+" if change_abs >= 0 else "-"
    pct_str = format_trend_pct(trend_pct)
    if trend_pct > 0:
        return (
            f'<span style="color:#1a8754;">▲ Up {pct_str} ({sign}{abs_str} per 1,000)</span> '
            f'<span style="color:#666;">since 2019</span>'
        )
    if trend_pct < 0:
        return (
            f'<span style="color:#c1372f;">▼ Down {pct_str} ({sign}{abs_str} per 1,000)</span> '
            f'<span style="color:#666;">since 2019</span>'
        )
    return '<span style="color:#666;">→ Unchanged since 2019</span>'


def compute_cmax(industry, include_col):
    values = by_industry[
        by_industry["industry_division"].eq(industry)
        & by_industry["year"].eq(LATEST_YEAR)
        & by_industry[include_col]
    ]["non_employing_per_1000_working_age"].dropna()
    if values.empty:
        return 1.0
    cmax = float(values.quantile(0.95))
    return round(max(cmax, 1.0), 1)

melbourne_rate_lookup = {}
melbourne_rate_rows = []
for industry in INDUSTRY_ORDER:
    for view_key, view_info in VIEWS.items():
        include_col = view_info["include_col"]
        for year in YEARS:
            eligible = by_industry[
                by_industry["industry_division"].eq(industry)
                & by_industry["year"].eq(year)
                & by_industry[include_col]
            ]
            working_age_pop = eligible["working_age_pop"].dropna().sum()
            business_count = eligible["non_employing_count"].dropna().sum()
            melbourne_rate = float(business_count) / float(working_age_pop) * 1000 if working_age_pop else np.nan
            melbourne_rate_lookup[(industry, view_info["rate_view"], year)] = melbourne_rate
            melbourne_rate_rows.append({
                "industry": industry,
                "view": view_info["rate_view"],
                "year": int(year),
                "melbourne_rate": melbourne_rate,
            })
melbourne_rate_table = pd.DataFrame(melbourne_rate_rows)
expected_rates = len(INDUSTRY_ORDER) * len(VIEWS) * len(YEARS)
if len(melbourne_rate_table) != expected_rates:
    raise ValueError(f"Expected {expected_rates} Melbourne rate rows, got {len(melbourne_rate_table)}")

# Story Mode uses fixed editorial steps, but all annotation labels and metrics are computed from the map payload.
STORY_VIEW_NAME = "All Melbourne (incl. CBD)"
STORY_INCLUDE_COL = "include_in_alternate_view"
STORY_VIEW_KEYS = {
    "Residential": "residential",
    "All": "all_melb",
    "All Melbourne (incl. CBD)": "all_melb",
}
STORY_INDUSTRY_KEYS = {"ALL": "ALL"} | {name: code for code, name in ANZSIC_DIVISIONS.items()}
STORY_INNER_LGAS = {"Melbourne", "Yarra", "Port Phillip", "Stonnington", "Boroondara"}
STORY_CORRIDOR_LGAS = {"Wyndham", "Melton", "Hume", "Whittlesea", "Casey", "Cardinia", "Mitchell"}
STORY_START_YEAR = min(YEARS)


def story_cluster_name(row):
    if bool(row["is_cbd_excluded"]) or row["council"] in STORY_INNER_LGAS:
        return "CBD + inner suburbs"
    if row["council"] in STORY_CORRIDOR_LGAS:
        return "Growth corridors"
    return "Middle ring suburbs"


by_industry["story_cluster"] = by_industry.apply(story_cluster_name, axis=1)

centroid_points = sa2_gdf.to_crs(epsg=7855).copy()
centroid_points["geometry"] = centroid_points.geometry.centroid
centroid_points = centroid_points.to_crs(epsg=4326)
sa2_centroids_dict = {
    row.SA2_NAME_2021: [round(float(row.geometry.x), 6), round(float(row.geometry.y), 6)]
    for row in centroid_points.itertuples(index=False)
}


def story_rows(cluster, industry, year):
    return by_industry[
        by_industry["story_cluster"].eq(cluster)
        & by_industry["industry_division"].eq(industry)
        & by_industry["year"].eq(int(year))
        & by_industry[STORY_INCLUDE_COL]
    ]


def story_count(cluster, industry, year):
    return float(story_rows(cluster, industry, year)["non_employing_count"].sum())


def story_rate(cluster, industry, year):
    rows = story_rows(cluster, industry, year)
    pop = rows["working_age_pop"].sum()
    if pop == 0:
        return np.nan
    return float(rows["non_employing_count"].sum()) / float(pop) * 1000


def story_growth(cluster, industry):
    return story_count(cluster, industry, LATEST_YEAR) - story_count(cluster, industry, STORY_START_YEAR)


def story_growth_pct(cluster, industry):
    start = story_count(cluster, industry, STORY_START_YEAR)
    return story_growth(cluster, industry) / start * 100 if start else np.nan


def story_cluster_total(cluster, year):
    return story_count(cluster, "ALL", year)


def story_industry_share(cluster, industry, year=LATEST_YEAR):
    total = story_cluster_total(cluster, year)
    return story_count(cluster, industry, year) / total * 100 if total else np.nan


def story_combined_share(cluster, industries, year=LATEST_YEAR):
    total = story_cluster_total(cluster, year)
    count = sum(story_count(cluster, industry, year) for industry in industries)
    return count / total * 100 if total else np.nan


def story_lq(cluster, industry, year=LATEST_YEAR):
    metro_industry = by_industry[
        by_industry["industry_division"].eq(industry)
        & by_industry["year"].eq(year)
        & by_industry[STORY_INCLUDE_COL]
    ]["non_employing_count"].sum()
    metro_total = by_industry[
        by_industry["industry_division"].eq("ALL")
        & by_industry["year"].eq(year)
        & by_industry[STORY_INCLUDE_COL]
    ]["non_employing_count"].sum()
    metro_share = metro_industry / metro_total if metro_total else np.nan
    local_share = story_industry_share(cluster, industry, year) / 100
    return local_share / metro_share if metro_share else np.nan


def story_growth_share(cluster, industry):
    metro_rows = by_industry[
        by_industry["industry_division"].eq(industry)
        & by_industry[STORY_INCLUDE_COL]
    ]
    metro = metro_rows.groupby("year")["non_employing_count"].sum()
    metro_growth = float(metro.loc[LATEST_YEAR] - metro.loc[STORY_START_YEAR])
    return story_growth(cluster, industry) / metro_growth * 100 if metro_growth else np.nan


def story_metric(label, value, detail):
    return {"label": label, "value": value, "detail": detail}


def story_pct(value):
    return f"{value:.0f}%" if not pd.isna(value) else "n/a"


def story_int(value):
    return f"{int(round(value)):,}" if not pd.isna(value) else "n/a"


def story_rate_label(value):
    return f"{value:.0f} per 1,000" if not pd.isna(value) else "n/a"


def story_sa2_density(sa2_name, industry, year=LATEST_YEAR, view_name=STORY_VIEW_NAME):
    view_key = STORY_VIEW_KEYS.get(view_name)
    industry_code = STORY_INDUSTRY_KEYS.get(industry, industry)
    if view_key not in VIEWS:
        raise ValueError(f"Unknown Story Mode view: {view_name}")
    if industry_code not in INDUSTRY_LABELS:
        raise ValueError(f"Unknown Story Mode industry: {industry}")
    include_col = VIEWS[view_key]["include_col"]
    rows = by_industry[
        by_industry["sa2_name"].eq(sa2_name)
        & by_industry["industry_division"].eq(industry_code)
        & by_industry["year"].eq(int(year))
        & by_industry[include_col]
    ]
    if rows.empty:
        raise ValueError(f"Missing Story Mode density for {sa2_name} / {industry} / {view_name} / {year}")
    return float(rows.iloc[0]["non_employing_per_1000_working_age"])


def story_top10_sa2_codes(cluster, industry, metric):
    industry_code = STORY_INDUSTRY_KEYS.get(industry, industry)
    rows = by_industry[
        by_industry["story_cluster"].eq(cluster)
        & by_industry["industry_division"].eq(industry_code)
        & by_industry[STORY_INCLUDE_COL]
    ].copy()
    if metric == "density_2024":
        ranked = rows[rows["year"].eq(LATEST_YEAR)].sort_values(
            ["non_employing_per_1000_working_age", "non_employing_count", "sa2_name"],
            ascending=[False, False, True],
        )
        return ranked.head(10)["sa2_code"].astype(str).tolist()
    if metric == "growth_abs":
        wide = rows[rows["year"].isin([STORY_START_YEAR, LATEST_YEAR])].pivot_table(
            index="sa2_code",
            columns="year",
            values="non_employing_count",
            aggfunc="sum",
            fill_value=0,
        )
        for year in [STORY_START_YEAR, LATEST_YEAR]:
            if year not in wide.columns:
                wide[year] = 0
        wide["growth_abs"] = wide[LATEST_YEAR] - wide[STORY_START_YEAR]
        ranked = wide.sort_values(["growth_abs", LATEST_YEAR], ascending=[False, False])
        return ranked.head(10).index.astype(str).tolist()
    raise ValueError(f"Unknown Story Mode top-10 metric: {metric}")


def story_top10_overlap_count(left_industry, left_metric, right_industry, right_metric, cluster="Growth corridors"):
    left = set(story_top10_sa2_codes(cluster, left_industry, left_metric))
    right = set(story_top10_sa2_codes(cluster, right_industry, right_metric))
    return len(left.intersection(right))


def story_sa2_count_label(value):
    return f"{value} SA2{'' if value == 1 else 's'}"


story_transport_construction_overlap = story_top10_overlap_count(
    "E", "density_2024", "I", "growth_abs"
)

story_total_2024 = by_industry[
    by_industry["industry_division"].eq("ALL")
    & by_industry["year"].eq(LATEST_YEAR)
    & by_industry[STORY_INCLUDE_COL]
]["non_employing_count"].sum()

STORY_STEP_DEFS = [
    {
        "id": 1,
        "title": "Melbourne has three self-employment economies",
        "caption": "Start with the whole map, including the CBD. We group Melbourne into three story zones: central + inner is the CBD plus the Melbourne, Yarra, Port Phillip, Stonnington and Boroondara council areas; the middle ring is the established suburban band between the inner cluster and the fringe; growth corridors are Wyndham, Melton, Hume, Whittlesea, Casey, Cardinia and Mitchell. Central + inner has the highest per-resident concentration, the middle ring holds the biggest number of self-employed businesses, and growth corridors are where the fastest new growth is happening.",
        "view": STORY_VIEW_NAME,
        "industry": "ALL",
        "year": 2024,
        "mapState": {"center": {"lat": -37.86, "lon": 144.96}, "zoom": 8.6},
        "metrics": [
            story_metric("Central + inner (incl. CBD)", story_rate_label(story_rate("CBD + inner suburbs", "ALL", LATEST_YEAR)), "highest cluster rate"),
            story_metric("Middle-ring base", story_int(story_cluster_total("Middle ring suburbs", LATEST_YEAR)), "businesses in 2024"),
            story_metric("Corridor growth", story_pct(story_growth_pct("Growth corridors", "ALL")), "increase since 2019"),
        ],
        "annotations": [
            {"sa2": "Melbourne CBD - West", "label_name": "CBD West"},
            {"sa2": "Dandenong - South", "label_name": "Dandenong South"},
            {"sa2": "Tarneit - North", "label_name": "Tarneit North"},
        ],
    },
    {
        "id": 2,
        "title": "The CBD and inner suburbs are a knowledge-work cluster",
        "caption": "This central + inner cluster is not mainly a delivery-work story. It leans toward professional services, real estate and finance: consultants, designers, lawyers, advisers and property operators working from offices, home offices or CBD registrations. CBD values can look very high because many businesses register in the central city while the resident denominator is smaller, so the Residential view removes those SA2s when you want a cleaner suburb-to-suburb comparison.",
        "view": STORY_VIEW_NAME,
        "industry": "Professional, Scientific and Technical Services",
        "year": 2024,
        "mapState": {"center": {"lat": -37.83, "lon": 144.975}, "zoom": 11.7},
        "metrics": [
            story_metric("Knowledge/property/finance", story_pct(story_combined_share("CBD + inner suburbs", ["M", "L", "K"])), "of central + inner total"),
            story_metric("Professional services", f"{story_lq('CBD + inner suburbs', 'M'):.1f}x", "the all-Melbourne share"),
            story_metric("CBD marker", story_rate_label(story_sa2_density("Melbourne CBD - West", "Professional, Scientific and Technical Services", LATEST_YEAR)), "Melbourne CBD West"),
        ],
        "annotations": [
            {"sa2": "Melbourne CBD - West", "label_name": "CBD West"},
            {"sa2": "South Melbourne", "label_name": "South Melbourne"},
            {"sa2": "Toorak", "label_name": "Toorak"},
        ],
    },
    {
        "id": 3,
        "title": "The middle ring is the quiet giant",
        "caption": "The middle ring means Melbourne's established suburban band between the inner-city councils and the outer growth corridors. It matters because it is broad and mixed rather than dominated by one industry. It holds about half of all self-employed businesses in the All Melbourne view, and it captured a large share of new health-care, professional-services and real-estate self-employment.",
        "view": STORY_VIEW_NAME,
        "industry": "ALL",
        "year": 2024,
        "mapState": {"center": {"lat": -37.93, "lon": 145.07}, "zoom": 9.9},
        "metrics": [
            story_metric("All-Melbourne share", story_pct(story_cluster_total("Middle ring suburbs", LATEST_YEAR) / story_total_2024 * 100), "of 2024 businesses"),
            story_metric("Health-care growth", story_pct(story_growth_share("Middle ring suburbs", "Q")), "of metro increase"),
            story_metric("Pro-services growth", story_pct(story_growth_share("Middle ring suburbs", "M")), "of metro increase"),
        ],
        "annotations": [
            {"sa2": "Dandenong - South", "label_name": "Dandenong South"},
            {"sa2": "Moorabbin - Heatherton", "label_name": "Moorabbin"},
            {"sa2": "Brighton (Vic.)", "label_name": "Brighton"},
        ],
    },
    {
        "id": 4,
        "title": "Outer growth corridors are the delivery-work shock",
        "caption": "Growth corridors are the fast-growing outer council areas on Melbourne's fringe. Transport, Postal and Warehousing is their clearest signal: rideshare drivers, delivery riders and parcel couriers. Wyndham dominates the highest-density SA2s, and corridor LGAs absorbed most of the all-Melbourne increase since 2019.",
        "view": STORY_VIEW_NAME,
        "industry": "Transport, Postal and Warehousing",
        "year": 2024,
        "mapState": {"center": {"lat": -37.83, "lon": 144.68}, "zoom": 11.0},
        "metrics": [
            story_metric("Transport share", story_pct(story_industry_share("Growth corridors", "I")), "of corridor total"),
            story_metric("Metro growth captured", story_pct(story_growth_share("Growth corridors", "I")), "since 2019"),
            story_metric("Corridor transport growth", story_pct(story_growth_pct("Growth corridors", "I")), "2019 to 2024"),
        ],
        "annotations": [
            {"sa2": "Tarneit - North", "label_name": "Tarneit North"},
            {"sa2": "Truganina - South West", "label_name": "Truganina SW"},
            {"sa2": "Tarneit - Central", "label_name": "Tarneit Central"},
        ],
    },
    {
        "id": 5,
        "title": "Construction is a different corridor economy",
        "caption": "Construction is also a growth-corridor story, but it lights up different places from delivery work. The highest-density construction SA2s do not overlap with the Transport-PSW SA2s that added the most self-employed businesses since 2019, so the map is showing separate local labour-market systems rather than one generic outer-suburb pattern.",
        "view": STORY_VIEW_NAME,
        "industry": "Construction",
        "year": 2024,
        "mapState": {"center": {"lat": -38.00, "lon": 145.27}, "zoom": 11.0},
        "metrics": [
            story_metric("Construction growth", story_pct(story_growth_share("Growth corridors", "E")), "captured by corridors"),
            story_metric("Top-10 overlap", story_sa2_count_label(story_transport_construction_overlap), "vs transport growth top 10"),
            story_metric("Corridor increase", story_int(story_growth("Growth corridors", "E")), "businesses since 2019"),
        ],
        "annotations": [
            {"sa2": "Narre Warren North", "label_name": "Narre Warren North"},
            {"sa2": "Doveton", "label_name": "Doveton"},
            {"sa2": "Hallam", "label_name": "Hallam"},
        ],
    },
]

missing_story_annotations = []

def story_annotation(sa2_name, label_name, industry_label, view_label, year):
    view_key = STORY_VIEW_KEYS.get(view_label)
    industry_code = STORY_INDUSTRY_KEYS.get(industry_label, industry_label)
    if view_key not in VIEWS:
        missing_story_annotations.append(f"{view_label}: unknown view")
        return None
    if industry_code not in INDUSTRY_LABELS:
        missing_story_annotations.append(f"{industry_label}: unknown industry")
        return None
    if sa2_name not in sa2_centroids_dict:
        missing_story_annotations.append(f"{sa2_name}: missing centroid")
        return None
    include_col = VIEWS[view_key]["include_col"]
    rows = by_industry[
        (by_industry["industry_division"] == industry_code)
        & (by_industry["year"] == int(year))
        & (by_industry["sa2_name"] == sa2_name)
        & (by_industry[include_col])
    ]
    if rows.empty:
        missing_story_annotations.append(
            f"{sa2_name}: missing density for {industry_label} / {view_label} / {year}"
        )
        return None
    density = float(rows.iloc[0]["non_employing_per_1000_working_age"])
    return {"sa2": sa2_name, "label": f"{label_name}\n{density:.0f} per 1,000"}

STORY_STEPS = []
for step in STORY_STEP_DEFS:
    step_out = {k: v for k, v in step.items() if k != "annotations"}
    annotations = []
    for annotation in step["annotations"]:
        built = story_annotation(
            annotation["sa2"], annotation["label_name"], step["industry"], step["view"], step["year"]
        )
        if built is not None:
            annotations.append(built)
    step_out["annotations"] = annotations
    STORY_STEPS.append(step_out)

if missing_story_annotations:
    print("Story Mode annotation validation failed:")
    for item in missing_story_annotations:
        print(f"- {item}")
    raise ValueError("Story Mode annotation SA2s must match available centroids and density rows")


PLOTLY_HOVER_TEMPLATE = (
    "<b>%{customdata[0]}</b><br>"
    "<span style='color:#666;font-size:12px;'>%{customdata[1]}</span>"
    "<br><br>"
    "<span style='font-size:15px;'>%{customdata[6]}</span><br>"
    "<span style='color:#666;font-size:12px;'>%{customdata[2]} · %{customdata[8]}</span>"
    "<br><br>"
    "%{customdata[7]}"
    "<br><br>"
    "<span style='color:#666;font-size:12px;'>"
    "%{customdata[4]:,} self-employed businesses<br>"
    "%{customdata[3]:,} working-age residents<br>"
    "Locally: %{customdata[5]} per 1,000<br>"
    "Greater Melbourne average: %{customdata[9]} per 1,000"
    "</span>"
    "<extra></extra>"
)

industry_payload = {}
for industry in INDUSTRY_ORDER:
    industry_payload[industry] = {
        "label": INDUSTRY_LABELS[industry],
        "views": {},
    }
    for view_key, view_info in VIEWS.items():
        include_col = view_info["include_col"]
        cmax = compute_cmax(industry, include_col)
        industry_payload[industry]["views"][view_key] = {"cmax": cmax, "years": {}}
        for year in YEARS:
            subset = by_industry[
                by_industry["industry_division"].eq(industry)
                & by_industry["year"].eq(year)
                & by_industry[include_col]
            ].sort_values("sa2_code")
            customdata = []
            for row in subset.itertuples(index=False):
                density = clean_number(row.non_employing_per_1000_working_age)
                melbourne_rate = melbourne_rate_lookup[(industry, view_info["rate_view"], year)]
                multiplier = multiplier_for(density, melbourne_rate)
                customdata.append([
                    row.sa2_name,
                    row.council,
                    tooltip_industry_label(industry),
                    clean_int(row.working_age_pop),
                    clean_int(row.non_employing_count),
                    format_density(density),
                    multiplier_label_for(multiplier),
                    trend_arrow_html_for(year, row.change_abs_since_2019, row.change_pct_since_2019),
                    int(year),
                    format_density(melbourne_rate),
                ])
            industry_payload[industry]["views"][view_key]["years"][str(year)] = {
                "locations": subset["sa2_code"].astype(str).tolist(),
                "z": [clean_number(value) for value in subset["non_employing_per_1000_working_age"].tolist()],
                "customdata": customdata,
            }

INDUSTRY_OPTION_COUNT = len(INDUSTRY_ORDER)
RESIDENTIAL_CMAX = industry_payload["ALL"]["views"]["residential"]["cmax"]
ALL_MELB_CMAX = industry_payload["ALL"]["views"]["all_melb"]["cmax"]

plotly_js = get_plotlyjs()
geojson_json = json.dumps(sa2_geojson, separators=(",", ":"))
payload_json = json.dumps(industry_payload, separators=(",", ":"))
industry_order_json = json.dumps(INDUSTRY_ORDER)
years_json = json.dumps(YEARS)
view_labels_json = json.dumps({key: value["label"] for key, value in VIEWS.items()})
hover_template_json = json.dumps(PLOTLY_HOVER_TEMPLATE, ensure_ascii=False)
centroids_json = json.dumps(sa2_centroids_dict, ensure_ascii=False, separators=(",", ":"))
story_steps_json = json.dumps(STORY_STEPS, ensure_ascii=False, separators=(",", ":"))
story_industry_keys_json = json.dumps(STORY_INDUSTRY_KEYS, ensure_ascii=False, separators=(",", ":"))
story_view_keys_json = json.dumps(STORY_VIEW_KEYS, ensure_ascii=False, separators=(",", ":"))
residential_exclusion_note = "; ".join(CBD_EXCLUSION_SA2_NAMES)

industry_options_html = "\n".join(
    f'        <option value="{industry}">{INDUSTRY_LABELS[industry]}</option>'
    for industry in INDUSTRY_ORDER
)
sa2_search_options_html = "\n".join(
    f'    <option value="{html_escape(sa2_name)}"></option>'
    for sa2_name in sorted(sa2_centroids_dict)
)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>Where is Melbourne's self-employed economy growing?</title>
<style>
  body {{ margin: 0; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #222; background: #fff; }}
  header {{ padding: 16px 24px 8px; }}
  header h1 {{ margin: 0 0 4px; font-size: 21px; }}
  header .subtitle {{ margin: 0; color: #555; font-size: 13px; max-width: 1050px; line-height: 1.35; }}
  .controls {{ padding: 6px 24px 8px; display: flex; align-items: center; gap: 12px; flex-wrap: wrap; font-size: 13px; }}
  .controls label {{ color: #444; font-weight: 600; }}
  .controls select {{ font-size: 13px; padding: 5px 8px; border: 1px solid #bbb; border-radius: 4px; background: #fafafa; }}
  #industry-select {{ min-width: 370px; }}
  .sa2-search {{ display: flex; align-items: center; gap: 6px; flex-wrap: wrap; }}
  #sa2-search-input {{ width: 210px; font-size: 13px; padding: 5px 8px; border: 1px solid #bbb; border-radius: 4px; background: #fff; }}
  .search-button {{ padding: 6px 10px; font-size: 13px; border: 1px solid #ccc; background: white; border-radius: 4px; cursor: pointer; }}
  .search-button:hover {{ background: #f0f0f0; }}
  .search-status {{ min-width: 160px; max-width: 340px; color: #555; font-size: 12px; line-height: 1.25; }}
  .view-note {{ margin: 0 24px 10px; color: #555; font-size: 12px; line-height: 1.45; max-width: 1180px; }}
  .view-note strong {{ color: #333; }}
  .map-panel {{ width: 100%; }}
  .year-controls {{ padding: 0 24px 4px; display: flex; align-items: center; gap: 10px; font-size: 13px; }}
  .year-controls button {{ padding: 4px 10px; border: 1px solid #bbb; border-radius: 4px; background: #f8f8f8; cursor: pointer; }}
  .year-controls input[type="range"] {{ width: 320px; }}
  .year-label {{ font-weight: 700; min-width: 42px; }}
  #story-panel {{
    display: none;
    background: #f7f9fc;
    border-left: 4px solid #1a4480;
    padding: 16px 20px;
    margin: 8px 24px 10px;
    font-family: inherit;
  }}
  .story-step-counter {{ font-size: 11px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; }}
  .story-step-title {{ margin: 4px 0 8px 0; font-size: 18px; line-height: 1.25; color: #1a4480; }}
  .story-step-caption {{ margin: 0 0 12px 0; font-size: 14px; line-height: 1.5; color: #333; max-width: 920px; }}
  .story-metrics {{ display: flex; flex-wrap: wrap; gap: 8px; margin: 0 0 14px 0; }}
  .story-metric {{ min-width: 142px; max-width: 210px; padding: 8px 10px; background: white; border: 1px solid #d9e0ea; border-radius: 6px; }}
  .story-metric-label {{ font-size: 10px; line-height: 1.2; color: #667085; text-transform: uppercase; letter-spacing: 0.4px; }}
  .story-metric-value {{ margin-top: 2px; font-size: 18px; line-height: 1.15; font-weight: 700; color: #1a4480; }}
  .story-metric-detail {{ margin-top: 2px; font-size: 11px; line-height: 1.25; color: #555; }}
  .story-nav {{ display: flex; gap: 8px; }}
  .story-nav button {{ padding: 6px 14px; font-size: 13px; border: 1px solid #ccc; background: white; border-radius: 4px; cursor: pointer; }}
  .story-nav button:disabled {{ opacity: 0.4; cursor: not-allowed; }}
  .story-nav button:hover:not(:disabled) {{ background: #f0f0f0; }}
  .story-toggle {{ padding: 6px 12px; font-size: 13px; border: 1px solid #ccc; background: white; border-radius: 4px; cursor: pointer; }}
  .story-toggle.active {{ background: #1a4480; color: white; border-color: #1a4480; }}
  #map-residential, #map-all_melb {{ position: relative; }}
  .search-marker-layer {{ position: absolute; inset: 0; pointer-events: none; z-index: 18; }}
  .search-marker {{ position: absolute; width: 14px; height: 14px; border-radius: 50%; background: #1a4480; border: 2px solid #fff; box-shadow: 0 0 0 2px rgba(26, 68, 128, 0.35), 0 2px 5px rgba(0, 0, 0, 0.28); transform: translate(-50%, -50%); }}
  .search-marker::after {{ content: ""; position: absolute; inset: -9px; border: 2px solid rgba(26, 68, 128, 0.35); border-radius: 50%; animation: markerPulse 1.2s ease-out 2; }}
  @keyframes markerPulse {{ from {{ transform: scale(0.55); opacity: 0.9; }} to {{ transform: scale(1.8); opacity: 0; }} }}
  .story-annotation-layer {{ position: absolute; inset: 0; pointer-events: none; z-index: 20; transition: opacity 120ms ease; }}
  .story-annotation-layer.tooltip-active {{ opacity: 0; }}
  .story-annotation {{ position: absolute; transform: translate(-50%, calc(-100% - 10px)); background: white; border: 1px solid #222; border-radius: 4px; padding: 4px 6px; color: #222; font-size: 11px; line-height: 1.25; text-align: center; white-space: nowrap; box-shadow: 0 1px 3px rgba(0, 0, 0, 0.18); }}
  .story-annotation::after {{ content: ""; position: absolute; left: 50%; bottom: -7px; width: 10px; height: 10px; background: white; border-right: 1px solid #222; border-bottom: 1px solid #222; transform: translateX(-50%) rotate(45deg); }}
  .map-footer {{
    margin-top: 16px;
    margin-left: 24px;
    margin-right: 24px;
    padding-top: 12px;
    padding-bottom: 16px;
    border-top: 1px solid #eee;
    font-size: 11px;
    color: #666;
    line-height: 1.5;
    font-family: inherit;
  }}
  .map-footer p {{ margin: 2px 0; }}
  .map-footer a {{ color: #1a4480; text-decoration: underline; }}
  .map-footer a:hover {{ text-decoration: none; }}
</style>
<script>{plotly_js}</script>
</head>
<body>
<header>
  <h1>Where is Melbourne's self-employed economy growing?</h1>
  <p class="subtitle">Non-employing businesses per 1,000 working-age residents, by SA2 (2019-2024). Includes sole traders, freelancers, gig workers, and home-based businesses.</p>
</header>
<div class="controls">
  <label for="industry-select">Industry filter</label>
  <select id="industry-select">
{industry_options_html}
  </select>
  <label for="view-select">View</label>
  <select id="view-select">
    <option value="residential">Residential Melbourne</option>
    <option value="all_melb" selected>All Melbourne (incl. CBD)</option>
  </select>
  <button id="story-mode-toggle" class="story-toggle" type="button">📖 Story Mode</button>
  <div class="sa2-search">
    <label for="sa2-search-input">Find Suburb or Area</label>
    <input id="sa2-search-input" list="sa2-search-options" type="search" placeholder="Search by suburb or area" autocomplete="off" />
    <datalist id="sa2-search-options">
{sa2_search_options_html}
    </datalist>
    <button id="sa2-search-button" class="search-button" type="button">Find</button>
    <span id="sa2-search-status" class="search-status" aria-live="polite"></span>
  </div>
</div>
<p class="view-note"><strong>View note:</strong> Story Mode uses All Melbourne (incl. CBD). Residential Melbourne removes seven high-distortion SA2s for cleaner suburb-to-suburb comparison: {residential_exclusion_note}.</p>
<div id="story-panel" style="display:none;">
  <div class="story-step-counter">Step <span id="story-step-num">1</span> of <span id="story-step-total">5</span></div>
  <h3 class="story-step-title" id="story-step-title"></h3>
  <p class="story-step-caption" id="story-step-caption"></p>
  <div class="story-metrics" id="story-metrics"></div>
  <div class="story-nav">
    <button id="story-prev" type="button" disabled>← Previous</button>
    <button id="story-next" type="button">Next →</button>
  </div>
</div>

<div id="panel-residential" class="map-panel" style="display:none;">
  <div class="year-controls">
    <button type="button" data-view="residential" data-action="play">Play</button>
    <button type="button" data-view="residential" data-action="pause">Pause</button>
    <label for="year-slider-residential">Year</label>
    <input id="year-slider-residential" type="range" min="2019" max="{LATEST_YEAR}" step="1" value="{LATEST_YEAR}" data-view="residential" />
    <span id="year-label-residential" class="year-label">{LATEST_YEAR}</span>
  </div>
  <div id="map-residential" style="width:100%;height:760px;"></div>
</div>
<div id="panel-all_melb" class="map-panel">
  <div class="year-controls">
    <button type="button" data-view="all_melb" data-action="play">Play</button>
    <button type="button" data-view="all_melb" data-action="pause">Pause</button>
    <label for="year-slider-all_melb">Year</label>
    <input id="year-slider-all_melb" type="range" min="2019" max="{LATEST_YEAR}" step="1" value="{LATEST_YEAR}" data-view="all_melb" />
    <span id="year-label-all_melb" class="year-label">{LATEST_YEAR}</span>
  </div>
  <div id="map-all_melb" style="width:100%;height:760px;"></div>
</div>
<footer class="map-footer">
  <p>Created for Suburban Futures Research, MDAP at University of Melbourne</p>
  <p>Visualisation by Neil Shekhar, Suburban Futures team</p>
  <p>
    Datasets are
    <a href="https://www.abs.gov.au/statistics/economy/business-indicators/counts-australian-businesses-including-entries-and-exits/latest-release" target="_blank" rel="noopener">Counts of Australian Businesses, Including Entries and Exits (CABEE)</a>
    2017-2021, 2018-2022 and 2021-2025 releases, and
    <a href="https://www.abs.gov.au/statistics/people/population/regional-population-age-and-sex/latest-release" target="_blank" rel="noopener">Regional Population by Age and Sex 2001-2024</a>,
    both from the Australian Bureau of Statistics
  </p>
</footer>
<script>
const SA2_GEOJSON = {geojson_json};
const SA2_CENTROIDS = {centroids_json};
const INDUSTRY_DATA = {payload_json};
const INDUSTRY_ORDER = {industry_order_json};
const YEARS = {years_json};
const VIEW_LABELS = {view_labels_json};
const VIEW_DIVS = {{ residential: 'map-residential', all_melb: 'map-all_melb' }};
const state = {{ industry: 'ALL', activeView: 'all_melb', years: {{ residential: {LATEST_YEAR}, all_melb: {LATEST_YEAR} }}, timers: {{ residential: null, all_melb: null }} }};
const plotConfig = {{ responsive: true, displaylogo: false }};
const hoverTemplate = {hover_template_json};

function traceFor(view, year) {{
  const industry = state.industry;
  const viewPayload = INDUSTRY_DATA[industry].views[view];
  const yearPayload = viewPayload.years[String(year)];
  return {{
    type: 'choroplethmapbox',
    geojson: SA2_GEOJSON,
    featureidkey: 'properties.SA2_CODE_2021',
    locations: yearPayload.locations,
    z: yearPayload.z,
    colorscale: 'Viridis',
    reversescale: true,
    zmin: 0,
    zmax: viewPayload.cmax,
    marker: {{ line: {{ width: 0.3, color: 'white' }}, opacity: 0.85 }},
    customdata: yearPayload.customdata,
    hovertemplate: hoverTemplate,
    colorbar: {{ title: {{ text: 'Self-employed<br>per 1,000<br>working-age' }} }},
    name: `${{industry}}_${{view}}_${{year}}`,
    showscale: true
  }};
}}

// Per-view mapbox pan/zoom state. Plotly.react() replaces the full layout, so
// uirevision alone does not preserve mapbox.center/zoom across industry / year
// changes. We capture the current pan/zoom on every plotly_relayout event
// (fired after the user pans, zooms, or any programmatic relayout) and re-inject
// it on the next render. Listeners are attached lazily after the first render.
const DEFAULT_MAP_STATE = {{
  residential: {{ center: {{ lat: -37.85, lon: 144.95 }}, zoom: 8.3 }},
  all_melb:    {{ center: {{ lat: -37.85, lon: 144.95 }}, zoom: 8.3 }}
}};
const mapState = cloneObject(DEFAULT_MAP_STATE);
const STORY_STEPS = {story_steps_json};
const STORY_INDUSTRY_KEYS = {story_industry_keys_json};
const STORY_VIEW_KEYS = {story_view_keys_json};
let storyModeActive = false;
let currentStoryStep = 0;
let preStoryState = null;
let activeStoryAnnotations = [];
let activeSearchResult = null;

const listenersAttached = {{ residential: false, all_melb: false }};
function cloneObject(value) {{ return JSON.parse(JSON.stringify(value)); }}
function storyViewKey(label) {{ return STORY_VIEW_KEYS[label] || label; }}
function storyIndustryKey(label) {{ return STORY_INDUSTRY_KEYS[label] || label; }}
function escapeHTML(value) {{
  return String(value).replace(/&/g, '&amp;').replace(/</g, '&lt;').replace(/>/g, '&gt;').replace(/"/g, '&quot;').replace(/'/g, '&#39;');
}}
function normalizeSearchValue(value) {{
  return String(value || '').trim().toLowerCase();
}}
function matchSA2Search(value) {{
  const query = normalizeSearchValue(value);
  if (!query) return null;
  const names = Object.keys(SA2_CENTROIDS).sort((a, b) => a.localeCompare(b));
  return names.find((name) => normalizeSearchValue(name) === query)
    || names.find((name) => normalizeSearchValue(name).startsWith(query))
    || names.find((name) => normalizeSearchValue(name).includes(query))
    || null;
}}
function sa2AvailableInView(sa2Name, view) {{
  const yearPayload = INDUSTRY_DATA[state.industry].views[view].years[String(state.years[view])];
  return yearPayload.customdata.some((row) => row[0] === sa2Name);
}}
function ensureStoryAnnotationLayer(view) {{
  const mapDiv = document.getElementById(VIEW_DIVS[view]);
  let layer = mapDiv.querySelector('.story-annotation-layer');
  if (!layer) {{
    layer = document.createElement('div');
    layer.className = 'story-annotation-layer';
    mapDiv.appendChild(layer);
  }}
  return layer;
}}
function ensureSearchMarkerLayer(view) {{
  const mapDiv = document.getElementById(VIEW_DIVS[view]);
  let layer = mapDiv.querySelector('.search-marker-layer');
  if (!layer) {{
    layer = document.createElement('div');
    layer.className = 'search-marker-layer';
    mapDiv.appendChild(layer);
  }}
  return layer;
}}
function clearSearchMarker() {{
  Object.values(VIEW_DIVS).forEach((divId) => {{
    const layer = document.getElementById(divId).querySelector('.search-marker-layer');
    if (layer) layer.innerHTML = '';
  }});
}}
function clearSearchLocator(clearInput = true) {{
  activeSearchResult = null;
  clearSearchMarker();
  document.getElementById('sa2-search-status').textContent = '';
  if (clearInput) document.getElementById('sa2-search-input').value = '';
}}
function drawSearchMarker() {{
  clearSearchMarker();
  if (!activeSearchResult || activeSearchResult.view !== state.activeView) {{
    document.getElementById('sa2-search-status').textContent = '';
    return;
  }}
  const gd = document.getElementById(VIEW_DIVS[state.activeView]);
  const map = gd && gd._fullLayout && gd._fullLayout.mapbox && gd._fullLayout.mapbox._subplot && gd._fullLayout.mapbox._subplot.map;
  if (!map || typeof map.project !== 'function') {{
    window.setTimeout(drawSearchMarker, 80);
    return;
  }}
  const centroid = SA2_CENTROIDS[activeSearchResult.sa2];
  if (!centroid) return;
  const point = map.project(centroid);
  const marker = document.createElement('div');
  marker.className = 'search-marker';
  marker.title = activeSearchResult.sa2;
  marker.style.left = point.x + 'px';
  marker.style.top = point.y + 'px';
  ensureSearchMarkerLayer(state.activeView).appendChild(marker);
}}
function clearMapAnnotations() {{
  Object.values(VIEW_DIVS).forEach((divId) => {{
    const layer = document.getElementById(divId).querySelector('.story-annotation-layer');
    if (layer) {{
      layer.innerHTML = '';
      layer.classList.remove('tooltip-active');
    }}
  }});
}}
function setStoryAnnotationsTooltipActive(view, isActive) {{
  const mapDiv = document.getElementById(VIEW_DIVS[view]);
  const layer = mapDiv && mapDiv.querySelector('.story-annotation-layer');
  if (layer) layer.classList.toggle('tooltip-active', Boolean(isActive));
}}
function renderMapAnnotations(annotations) {{
  activeStoryAnnotations = annotations || [];
  drawStoryAnnotations();
}}
function drawStoryAnnotations() {{
  clearMapAnnotations();
  if (!storyModeActive || !activeStoryAnnotations.length) return;
  const view = state.activeView;
  const gd = document.getElementById(VIEW_DIVS[view]);
  const map = gd && gd._fullLayout && gd._fullLayout.mapbox && gd._fullLayout.mapbox._subplot && gd._fullLayout.mapbox._subplot.map;
  if (!map || typeof map.project !== 'function') {{
    window.setTimeout(drawStoryAnnotations, 80);
    return;
  }}
  const layer = ensureStoryAnnotationLayer(view);
  activeStoryAnnotations.forEach((annotation) => {{
    const centroid = SA2_CENTROIDS[annotation.sa2];
    if (!centroid) return;
    const point = map.project(centroid);
    const label = document.createElement('div');
    label.className = 'story-annotation';
    label.style.left = point.x + 'px';
    label.style.top = point.y + 'px';
    label.innerHTML = escapeHTML(annotation.label).replace(/\\n/g, '<br>');
    layer.appendChild(label);
  }});
}}
function syncControlsFromState() {{
  document.getElementById('industry-select').value = state.industry;
  document.getElementById('view-select').value = state.activeView;
  Object.keys(VIEW_DIVS).forEach((view) => {{
    document.getElementById(`year-slider-${{view}}`).value = state.years[view];
    document.getElementById(`year-label-${{view}}`).textContent = String(state.years[view]);
  }});
}}
function showActivePanel() {{
  Object.keys(VIEW_DIVS).forEach((view) => {{
    document.getElementById(`panel-${{view}}`).style.display = view === state.activeView ? 'block' : 'none';
  }});
}}
function pauseAll() {{ Object.keys(VIEW_DIVS).forEach((view) => pause(view)); }}
function deactivateStoryModeForUserOverride() {{
  if (!storyModeActive) return;
  storyModeActive = false;
  currentStoryStep = 0;
  preStoryState = null;
  activeStoryAnnotations = [];
  document.getElementById('story-panel').style.display = 'none';
  document.getElementById('story-mode-toggle').textContent = '📖 Story Mode';
  document.getElementById('story-mode-toggle').classList.remove('active');
  clearMapAnnotations();
}}
async function enterStoryMode() {{
  activeSearchResult = null;
  clearSearchMarker();
  document.getElementById('sa2-search-status').textContent = '';
  preStoryState = {{ industry: state.industry, activeView: state.activeView, years: cloneObject(state.years), mapState: cloneObject(mapState) }};
  storyModeActive = true;
  currentStoryStep = 0;
  pauseAll();
  document.getElementById('story-panel').style.display = 'block';
  document.getElementById('story-mode-toggle').textContent = '✕ Exit Story Mode';
  document.getElementById('story-mode-toggle').classList.add('active');
  await applyStoryStep(0);
}}
async function exitStoryMode() {{
  storyModeActive = false;
  activeStoryAnnotations = [];
  pauseAll();
  document.getElementById('story-panel').style.display = 'none';
  document.getElementById('story-mode-toggle').textContent = '📖 Story Mode';
  document.getElementById('story-mode-toggle').classList.remove('active');
  clearMapAnnotations();
  if (preStoryState) {{
    state.industry = preStoryState.industry;
    state.activeView = preStoryState.activeView;
    state.years = preStoryState.years;
    Object.keys(mapState).forEach((view) => {{ mapState[view] = preStoryState.mapState[view]; }});
    preStoryState = null;
    syncControlsFromState();
    showActivePanel();
    await renderAllViews();
    Plotly.Plots.resize(document.getElementById(VIEW_DIVS[state.activeView]));
  }}
}}
async function applyStoryStep(idx) {{
  const step = STORY_STEPS[idx];
  const view = storyViewKey(step.view);
  const industry = storyIndustryKey(step.industry);
  currentStoryStep = idx;
  pauseAll();
  document.getElementById('story-step-num').textContent = (idx + 1).toString();
  document.getElementById('story-step-total').textContent = STORY_STEPS.length.toString();
  document.getElementById('story-step-title').textContent = step.title;
  document.getElementById('story-step-caption').textContent = step.caption;
  const metricList = document.getElementById('story-metrics');
  metricList.innerHTML = '';
  (step.metrics || []).forEach((metric) => {{
    const item = document.createElement('div');
    item.className = 'story-metric';
    const label = document.createElement('div');
    label.className = 'story-metric-label';
    label.textContent = metric.label;
    const value = document.createElement('div');
    value.className = 'story-metric-value';
    value.textContent = metric.value;
    const detail = document.createElement('div');
    detail.className = 'story-metric-detail';
    detail.textContent = metric.detail;
    item.append(label, value, detail);
    metricList.appendChild(item);
  }});
  document.getElementById('story-prev').disabled = idx === 0;
  document.getElementById('story-next').disabled = idx === STORY_STEPS.length - 1;
  state.activeView = view;
  state.industry = industry;
  state.years[view] = step.year;
  mapState[view] = cloneObject(step.mapState);
  syncControlsFromState();
  showActivePanel();
  await renderAllViews();
  Plotly.Plots.resize(document.getElementById(VIEW_DIVS[state.activeView]));
  renderMapAnnotations(step.annotations);
}}
function attachMapListener(view) {{
  if (listenersAttached[view]) return;
  const div = document.getElementById(VIEW_DIVS[view]);
  if (!div || typeof div.on !== 'function') return;
  div.on('plotly_relayout', (e) => {{
    if (e['mapbox.center']) mapState[view].center = e['mapbox.center'];
    if (e['mapbox.zoom'] !== undefined) mapState[view].zoom = e['mapbox.zoom'];
    if (e.mapbox && e.mapbox.center) mapState[view].center = e.mapbox.center;
    if (e.mapbox && e.mapbox.zoom !== undefined) mapState[view].zoom = e.mapbox.zoom;
    if (storyModeActive && view === state.activeView) window.requestAnimationFrame(drawStoryAnnotations);
    if (activeSearchResult && activeSearchResult.view === view) window.requestAnimationFrame(drawSearchMarker);
  }});
  div.on('plotly_hover', () => setStoryAnnotationsTooltipActive(view, true));
  div.on('plotly_unhover', () => setStoryAnnotationsTooltipActive(view, false));
  div.addEventListener('mouseleave', () => setStoryAnnotationsTooltipActive(view, false));
  listenersAttached[view] = true;
}}
function layoutFor(view, year) {{
  return {{
    mapbox: {{
      style: 'carto-positron',
      center: mapState[view].center,
      zoom: mapState[view].zoom
    }},
    margin: {{ l: 0, r: 0, t: 10, b: 0 }},
    height: 760,
    transition: {{ duration: 250 }},
    hoverlabel: {{
      bgcolor: 'white',
      bordercolor: '#cccccc',
      font: {{ family: "-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif", size: 14, color: '#222222' }},
      align: 'left'
    }},
    uirevision: view
  }};
}}

async function renderView(view) {{
  const year = state.years[view];
  document.getElementById(`year-slider-${{view}}`).value = year;
  document.getElementById(`year-label-${{view}}`).textContent = String(year);
  await Plotly.react(VIEW_DIVS[view], [traceFor(view, year)], layoutFor(view, year), plotConfig);
  attachMapListener(view);
  if (storyModeActive && view === state.activeView) window.requestAnimationFrame(drawStoryAnnotations);
  if (activeSearchResult && view === state.activeView) window.requestAnimationFrame(drawSearchMarker);
}}

async function renderAllViews() {{
  await renderView('residential');
  await renderView('all_melb');
}}

async function setIndustry(industry) {{
  state.industry = industry;
  syncControlsFromState();
  await renderAllViews();
  if (storyModeActive) drawStoryAnnotations();
}}

async function setView(view) {{
  state.activeView = view;
  syncControlsFromState();
  showActivePanel();
  await renderView(view);
  Plotly.Plots.resize(document.getElementById(VIEW_DIVS[view]));
  if (storyModeActive) drawStoryAnnotations();
}}

async function setYear(view, year) {{
  state.years[view] = Number(year);
  syncControlsFromState();
  await renderView(view);
  if (storyModeActive && view === state.activeView) drawStoryAnnotations();
  if (activeSearchResult && view === state.activeView) drawSearchMarker();
}}

async function locateSA2FromSearch() {{
  const input = document.getElementById('sa2-search-input');
  const status = document.getElementById('sa2-search-status');
  if (!input.value.trim()) {{
    await resetSearchLocator();
    return;
  }}
  const match = matchSA2Search(input.value);
  if (!match) {{
    clearSearchLocator(false);
    status.textContent = 'No matching SA2 found';
    return;
  }}
  input.value = match;
  deactivateStoryModeForUserOverride();
  pauseAll();
  let targetView = state.activeView;
  let statusText = `Zoomed to ${{match}}`;
  if (!sa2AvailableInView(match, targetView) && sa2AvailableInView(match, 'all_melb')) {{
    targetView = 'all_melb';
    statusText = `${{match}} is shown in All Melbourne`;
  }} else if (!sa2AvailableInView(match, targetView)) {{
    clearSearchLocator(false);
    status.textContent = `${{match}} is outside the mapped views`;
    return;
  }}
  const [lon, lat] = SA2_CENTROIDS[match];
  state.activeView = targetView;
  mapState[targetView] = {{ center: {{ lat, lon }}, zoom: 11.8 }};
  activeSearchResult = {{ sa2: match, view: targetView }};
  syncControlsFromState();
  showActivePanel();
  await renderView(targetView);
  Plotly.Plots.resize(document.getElementById(VIEW_DIVS[targetView]));
  drawSearchMarker();
  status.textContent = statusText;
}}

async function resetSearchLocator() {{
  clearSearchLocator(false);
  mapState[state.activeView] = cloneObject(DEFAULT_MAP_STATE[state.activeView]);
  await renderView(state.activeView);
  Plotly.Plots.resize(document.getElementById(VIEW_DIVS[state.activeView]));
}}

function pause(view) {{
  if (state.timers[view]) {{
    clearInterval(state.timers[view]);
    state.timers[view] = null;
  }}
}}

function play(view) {{
  pause(view);
  state.timers[view] = setInterval(async () => {{
    const current = state.years[view];
    const next = current >= {LATEST_YEAR} ? 2019 : current + 1;
    await setYear(view, next);
  }}, 900);
}}

function applyQueryParams() {{
  const params = new URLSearchParams(window.location.search);
  const industry = params.get('industry');
  const view = params.get('view');
  const year = params.get('year');
  if (industry && INDUSTRY_DATA[industry]) state.industry = industry;
  if (view && VIEW_DIVS[view]) state.activeView = view;
  if (year && YEARS.includes(Number(year))) {{
    state.years.residential = Number(year);
    state.years.all_melb = Number(year);
  }}
}}

function wireControls() {{
  document.getElementById('industry-select').addEventListener('change', async (event) => {{
    deactivateStoryModeForUserOverride();
    await setIndustry(event.target.value);
  }});
  document.getElementById('view-select').addEventListener('change', async (event) => {{
    deactivateStoryModeForUserOverride();
    clearSearchLocator();
    await setView(event.target.value);
  }});
  document.getElementById('sa2-search-button').addEventListener('click', locateSA2FromSearch);
  document.getElementById('sa2-search-input').addEventListener('keydown', async (event) => {{
    if (event.key === 'Enter') {{
      event.preventDefault();
      await locateSA2FromSearch();
    }}
  }});
  document.getElementById('sa2-search-input').addEventListener('input', async (event) => {{
    if (!event.target.value.trim()) await resetSearchLocator();
  }});
  document.getElementById('sa2-search-input').addEventListener('change', locateSA2FromSearch);
  document.getElementById('story-mode-toggle').addEventListener('click', async () => {{
    if (storyModeActive) await exitStoryMode();
    else await enterStoryMode();
  }});
  document.getElementById('story-prev').addEventListener('click', async () => {{
    if (currentStoryStep > 0) await applyStoryStep(currentStoryStep - 1);
  }});
  document.getElementById('story-next').addEventListener('click', async () => {{
    if (currentStoryStep < STORY_STEPS.length - 1) await applyStoryStep(currentStoryStep + 1);
  }});
  for (const slider of document.querySelectorAll('input[type="range"][data-view]')) {{
    slider.addEventListener('input', async (event) => {{
      deactivateStoryModeForUserOverride();
      await setYear(event.target.dataset.view, event.target.value);
    }});
  }}
  for (const button of document.querySelectorAll('button[data-view][data-action]')) {{
    button.addEventListener('click', (event) => {{
      deactivateStoryModeForUserOverride();
      const view = event.target.dataset.view;
      const action = event.target.dataset.action;
      if (action === 'play') play(view);
      if (action === 'pause') pause(view);
    }});
  }}
}}

window.selfEmployedSetIndustry = setIndustry;
window.selfEmployedSetView = setView;
window.selfEmployedSetYear = setYear;
window.selfEmployedLocateSA2 = async function (sa2Name) {{
  document.getElementById('sa2-search-input').value = sa2Name;
  await locateSA2FromSearch();
}};
window.selfEmployedEnterStoryMode = enterStoryMode;
window.selfEmployedExitStoryMode = exitStoryMode;
window.selfEmployedApplyStoryStep = applyStoryStep;
window.selfEmployedState = state;
window.selfEmployedExpandIndustryDropdownForScreenshot = function () {{
  const select = document.getElementById('industry-select');
  select.size = INDUSTRY_ORDER.length;
  select.style.width = '520px';
  select.style.maxHeight = '520px';
}};

applyQueryParams();
wireControls();
setIndustry(state.industry).then(() => setView(state.activeView)).then(() => {{
  const params = new URLSearchParams(window.location.search);
  if (params.get('dropdown') === 'open') window.selfEmployedExpandIndustryDropdownForScreenshot();
}});
</script>
</body>
</html>
"""

HTML_OUT.write_text(html, encoding="utf-8")
html_size_mb = HTML_OUT.stat().st_size / (1024 * 1024)
if html_size_mb > 80:
    raise ValueError(f"HTML is too large ({html_size_mb:.2f} MB); JSON payload needs a different approach")

print(f"Wrote HTML: {HTML_OUT} ({html_size_mb:.2f} MB)")
print(f"Industry dropdown options: {INDUSTRY_OPTION_COUNT}")
print("Industry dropdown order:")
for industry in INDUSTRY_ORDER:
    print(f"  {industry}: {INDUSTRY_LABELS[industry]}")
print(f"Aggregate residential cmax: {RESIDENTIAL_CMAX:.1f}")
print(f"Aggregate all-melb cmax: {ALL_MELB_CMAX:.1f}")


### Write Updated Static Preview PNG

The preview PNG mirrors the default All Melbourne 2024 aggregate view. It is generated from the Plotly figure only; the surrounding HTML controls, Story Mode panel, and footer remain HTML-only.


In [ ]:
import plotly.graph_objects as go

PNG_OUT = OUT_DIR / "self_employed_business_density_2024_preview.png"
aggregate_latest = by_industry[
    by_industry["industry_division"].eq("ALL")
    & by_industry["year"].eq(LATEST_YEAR)
    & by_industry["include_in_alternate_view"]
].sort_values("sa2_code").copy()

png_customdata = []
for row in aggregate_latest.itertuples(index=False):
    density = clean_number(row.non_employing_per_1000_working_age)
    melbourne_rate = melbourne_rate_lookup[("ALL", "All", LATEST_YEAR)]
    multiplier = multiplier_for(density, melbourne_rate)
    png_customdata.append([
        row.sa2_name,
        row.council,
        tooltip_industry_label("ALL"),
        clean_int(row.working_age_pop),
        clean_int(row.non_employing_count),
        format_density(density),
        multiplier_label_for(multiplier),
        trend_arrow_html_for(LATEST_YEAR, row.change_abs_since_2019, row.change_pct_since_2019),
        int(LATEST_YEAR),
        format_density(melbourne_rate),
    ])

png_fig = go.Figure(data=[go.Choroplethmapbox(
    geojson=sa2_geojson,
    featureidkey="properties.SA2_CODE_2021",
    locations=aggregate_latest["sa2_code"],
    z=aggregate_latest["non_employing_per_1000_working_age"],
    colorscale="Viridis",
    reversescale=True,
    zmin=0,
    zmax=ALL_MELB_CMAX,
    marker_line_width=0.3,
    marker_line_color="white",
    marker_opacity=0.85,
    customdata=png_customdata,
    hovertemplate=PLOTLY_HOVER_TEMPLATE,
    colorbar=dict(title="Self-employed<br>per 1,000<br>working-age"),
    showscale=True,
    name="ALL_all_melb_2024",
)])
png_fig.update_layout(
    mapbox_style="carto-positron",
    mapbox_center={"lat": -37.85, "lon": 144.95},
    mapbox_zoom=8.3,
    margin={"l": 0, "r": 0, "t": 86, "b": 70},
    title={
        "text": "<b>Where is Melbourne's self-employed economy growing?</b><br>"
                "<sub>Non-employing businesses per 1,000 working-age residents, by SA2 (2024). Includes sole traders, freelancers, gig workers, and home-based businesses.</sub>",
        "x": 0.5,
        "xanchor": "center",
    },
    hoverlabel=dict(
        bgcolor="white",
        bordercolor="#cccccc",
        font=dict(family="-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif", size=14, color="#222222"),
        align="left",
    ),
    annotations=[{
        "text": (
            "'Self-employed businesses' means non-employing businesses: registered ABNs with no payroll employees. "
            "Use the industry filter in the HTML map to separate gig-economy corridor growth from professional-services inner-suburb concentration. "
            "Source: ABS CABEE 2017-2025 releases; map window 2019-2024; ABS ERP 2024."
        ),
        "showarrow": False,
        "xref": "paper", "yref": "paper",
        "x": 0.5, "y": -0.04,
        "xanchor": "center", "yanchor": "top",
        "font": {"size": 10, "color": "#555"},
    }],
)
png_fig.write_image(PNG_OUT, width=1600, height=1000, scale=2)
print(f"Wrote PNG: {PNG_OUT} ({PNG_OUT.stat().st_size / 1024:.0f} KB)")


### Final Verification

This final cell prints the required build checks for the industry-filtered map: aggregation checks, file sizes, and dropdown option count/order.

In [ ]:
html_size_mb = HTML_OUT.stat().st_size / (1024 * 1024)
png_size_kb = PNG_OUT.stat().st_size / 1024
print("=== Industry filter map verification ===")
for year in [2019, 2022, LATEST_YEAR]:
    print(f"1. Industry sum equals aggregate for {year}, primary-view SA2s: {'PASS' if SUM_CHECK_RESULTS[year] else 'FAIL'}")
print(f"2. ALL pseudo-division exact match with self_employed_business_density.csv: {'PASS' if ALL_CHECK_RESULT else 'FAIL'}")
print(f"3. HTML file size: {html_size_mb:.2f} MB ({'PASS under 80 MB target' if html_size_mb < 80 else 'CHECK size target'})")
print("4. Story Mode data and UI are built into the HTML scaffold; verify interactions in Chrome after HTML build")
print(f"5. Industry dropdown option count: {INDUSTRY_OPTION_COUNT} ({'PASS' if INDUSTRY_OPTION_COUNT == 20 else 'FAIL'})")
print("Industry dropdown order:")
for i, industry in enumerate(INDUSTRY_ORDER, start=1):
    print(f"  {i:02d}. {INDUSTRY_LABELS[industry]}")
print(f"HTML output: {HTML_OUT} ({html_size_mb:.2f} MB)")
print(f"PNG output: {PNG_OUT} ({png_size_kb:.0f} KB)")

transport_rate_2024 = melbourne_rate_lookup[("I", "Residential", LATEST_YEAR)]
transport_top_multipliers = by_industry[
    by_industry["industry_division"].eq("I")
    & by_industry["year"].eq(LATEST_YEAR)
    & by_industry["include_in_primary_view"]
].copy()
transport_top_multipliers["melbourne_rate"] = transport_rate_2024
transport_top_multipliers["multiplier"] = (
    transport_top_multipliers["non_employing_per_1000_working_age"] / transport_rate_2024
).round(2)
transport_top_multipliers = transport_top_multipliers.sort_values(
    ["multiplier", "non_employing_per_1000_working_age", "non_employing_count"],
    ascending=[False, False, False],
).head(5)
print("\nTop 5 Transport-PSW 2024 Residential Melbourne SA2s by Greater Melbourne average multiplier:")
print(transport_top_multipliers[[
    "sa2_name", "council", "non_employing_per_1000_working_age", "melbourne_rate", "multiplier"
]].to_string(index=False))
